# Qwen3 distillation — corpus teacher cố định, Kaggle pilot

Upload notebook này lên Kaggle, bật **GPU + Internet**. Không cần clone repo hoặc GitHub token.
Source train đã được đóng gói. Notebook tải duy nhất file teacher Qwen v2; không tải/chạy teacher.

Mặc định: Qwen3-0.6B, 5.000 mẫu train tối đa sau lọc, 200 bước mỗi nhánh, LoRA rank 16,
sequence tối đa 2.048, micro-batch 1 × accumulation 8. Chạy **uniform** và **progress** tuần tự.
Đây là pilot để kiểm tra phương pháp, không phải tái lập paper hay kết quả benchmark cuối.
Đổi sang Qwen3-1.7B ở cell cấu hình để thử model thứ hai.

Pipeline: tải → audit → xác nhận mapping → prepare/split → kiểm tra runtime → train → so sánh log → export.
**Không Run All ngay lần đầu**: đọc dữ liệu tại bước audit và xác nhận prompt/response trước khi prepare.
Không có kết quả train Kaggle được xác nhận trước; thời gian/VRAM phụ thuộc GPU và token thực tế.


In [ ]:
from pathlib import Path
import json, os, sys, subprocess, hashlib, shutil
BASE = Path('/kaggle/working/distill-pilot' if Path('/kaggle').exists()
            else '/content/distill-pilot' if Path('/content').exists() else './distill-pilot').resolve()
BASE.mkdir(parents=True, exist_ok=True)
MODEL = 'Qwen/Qwen3-0.6B'  # Lần sau: 'Qwen/Qwen3-1.7B'
MODEL_REVISION = 'main'  # prepare sẽ resolve SHA cụ thể và lưu vào manifest
DATASET = 'zhang-yice/sentiment-distillation-v2'
DATASET_REVISION = '0ee48fc022bb45ad9f2c66c43fc1c73db6c3f4fb'
TEACHER_FILE = 'Qwen3-32B_50k.jsonl'
LOCAL_TEACHER_FILE = None  # Hoặc '/kaggle/input/.../Qwen3-32B_50k.jsonl'
INSTALL_DEPENDENCIES = True  # Giữ CUDA torch của Kaggle; có dry-run bảo vệ torch
FOUR_BIT = False  # LoRA thường trước; True thêm bitsandbytes và bật QLoRA
RUN_TAG = 'pilot-v1'  # Đổi tên nếu thay cấu hình; không ghi đè thí nghiệm cũ
MAX_TRAIN = 5000  # 0 = toàn bộ train hợp lệ sau grouped holdout
MAX_LENGTH = 2048  # Mẫu dài hơn bị loại nguyên mẫu, không cắt đáp án
STEPS = 200
MODES = ['uniform', 'progress']  # Ablation: thêm 'balanced', 'loss'
SEED = 42
GRAD_ACCUM = 8
LEARNING_RATE = 1e-4
PROBE_EVERY = 25
SAVE_EVERY = 100
RESUME = {}  # Ví dụ {'progress': '/kaggle/working/.../progress/checkpoint-100'}
EXPORT_MODE = None  # Đặt 'uniform' hoặc 'progress' sau khi train xong
MODEL_TAG = MODEL.split('/')[-1]
WORK = BASE / RUN_TAG / MODEL_TAG
WORK.mkdir(parents=True, exist_ok=True)
print('Work:', WORK)


In [ ]:
BUNDLED = {'training/distill_core.py': '"""Dependency-free data checks, splits and adaptive group weights."""\nimport hashlib\nimport json\nimport math\nimport random\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True).encode()).hexdigest()\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef read_records(path):\n    with open(path, encoding=\'utf-8-sig\') as stream:\n        for line, text in enumerate(stream, 1):\n            if not text.strip():\n                continue\n            try:\n                obj = json.loads(text)\n                if not isinstance(obj, dict):\n                    raise ValueError(\'record is not an object\')\n                yield line, obj\n            except (ValueError, TypeError) as exc:\n                raise ValueError(f\'Invalid JSON record at line {line}: {exc}\') from exc\n\n\ndef mapped_record(obj, prompt_fields, response_field, group_field, source_field):\n    """Explicit field order; preserve each field\'s contents, including whitespace."""\n    parts = []\n    for field in prompt_fields:\n        value = obj.get(field)\n        if value is None:\n            raise ValueError(f\'missing prompt field: {field}\')\n        if not isinstance(value, str):\n            raise ValueError(f\'prompt field must be a string: {field}\')\n        if value.strip():\n            parts.append(value)\n    response = obj.get(response_field)\n    if not parts or not isinstance(response, str) or not response.strip():\n        raise ValueError(\'empty prompt or response\')\n    group = obj.get(group_field)\n    if group is None or not str(group).strip():\n        raise ValueError(f\'missing group field: {group_field}\')\n    source = obj.get(source_field)\n    if not isinstance(source, str) or not source.strip():\n        raise ValueError(f\'missing source/input field for group split: {source_field}\')\n    # Normalize only the split key; never alter the training target.\n    source_key = \' \'.join(source.split()).casefold()\n    return {\'prompt\': \'\\n\\n\'.join(parts), \'response\': response,\n            \'group\': str(group), \'source_key\': digest(source_key)}\n\n\ndef split_records(records, seed, dev_fraction=0.1, max_train=0, probe_per_group=4,\n                  holdout_per_group=4, min_group_train=8, max_groups=32):\n    if not 0 < dev_fraction < 1:\n        raise ValueError(\'dev_fraction must be between 0 and 1\')\n    if min(probe_per_group, holdout_per_group, min_group_train, max_groups) < 1 or max_train < 0:\n        raise ValueError(\'Invalid split limits\')\n    # Assign every input group before selecting a pilot subset. No input crosses splits.\n    keys = sorted({r[\'source_key\'] for r in records})\n    random.Random(seed).shuffle(keys)\n    ndev = max(2, round(len(keys) * dev_fraction))\n    if ndev >= len(keys):\n        raise ValueError(\'Not enough independent inputs to split\')\n    dev_keys = set(keys[:ndev])\n    probe_keys = set(keys[:ndev // 2])\n    train = [r for r in records if r[\'source_key\'] not in dev_keys]\n    random.Random(seed).shuffle(train)\n    if max_train:\n        train = train[:max_train]\n    counts = Counter(r[\'group\'] for r in train)\n    # Group collapsing uses TRAIN only. Reserve one slot for rare groups.\n    common = [g for g, n in sorted(counts.items(), key=lambda x: (-x[1], x[0]))\n              if n >= min_group_train][:max_groups - 1]\n    mapping = {g: g if g in common else \'__rare__\' for g in counts}\n    groups = sorted(set(mapping.values()))\n    for r in train:\n        r[\'train_group\'] = mapping[r[\'group\']]\n    probe, holdout = [], []\n    for allowed, dest, limit in ((probe_keys, probe, probe_per_group),\n                                  (dev_keys - probe_keys, holdout, holdout_per_group)):\n        buckets = defaultdict(list)\n        for r in records:\n            if r[\'source_key\'] in allowed and r[\'group\'] in mapping:\n                r[\'train_group\'] = mapping[r[\'group\']]\n                buckets[r[\'train_group\']].append(r)\n        for g in groups:\n            items = sorted(buckets[g], key=lambda r: r[\'id\'])\n            random.Random(f\'{seed}:{g}\').shuffle(items)\n            dest.extend(items[:limit])\n    if not train or not probe or not holdout:\n        raise ValueError(\'Empty train/probe/holdout; increase pilot size or dev fraction\')\n    return train, probe, holdout, mapping\n\n\nclass GroupWeights:\n    """Progress normalized by initial per-response CE; bounded mean-one weights."""\n    def __init__(self, groups, counts, mode=\'progress\', mix=0.5, temperature=0.05,\n                 ema=0.5, floor=0.5, ceiling=2.0):\n        if mode not in (\'uniform\', \'balanced\', \'loss\', \'progress\'):\n            raise ValueError(\'Unknown weighting mode\')\n        if not 0 <= mix <= 1 or temperature <= 0 or not 0 <= ema < 1:\n            raise ValueError(\'Invalid mix/temperature/EMA\')\n        if not 0 < floor <= 1 <= ceiling:\n            raise ValueError(\'Require 0 < floor <= 1 <= ceiling\')\n        self.groups = list(groups)\n        self.counts = dict(counts)\n        self.mode, self.mix, self.temperature, self.ema = mode, mix, temperature, ema\n        self.floor, self.ceiling = floor, ceiling\n        self.initial, self.previous, self.smoothed = {}, {}, {}\n        self.weights = {g: 1.0 for g in groups}\n        if mode == \'balanced\':\n            self.weights = self._bounded({g: 1 / counts[g] for g in groups})\n\n    def _bounded(self, raw):\n        # Solve E_train[clip(scale * raw, floor, ceiling)] = 1.\n        total = sum(self.counts.values())\n        lo, hi = 0., 1.\n        def values(scale):\n            return {g: min(self.ceiling, max(self.floor, scale * raw[g])) for g in self.groups}\n        def mean(w):\n            return sum(w[g] * self.counts[g] for g in self.groups) / total\n        while mean(values(hi)) < 1 - 1e-12:\n            hi *= 2\n        for _ in range(80):\n            mid = (lo + hi) / 2\n            if mean(values(mid)) < 1:\n                lo = mid\n            else:\n                hi = mid\n        return values((lo + hi) / 2)\n\n    def update(self, losses):\n        if not losses or any(not math.isfinite(v) or v < 0 for v in losses.values()):\n            raise ValueError(\'Probe losses must be nonnegative and finite\')\n        scores = {}\n        for g, value in losses.items():\n            if g not in self.weights:\n                raise ValueError(f\'Unknown probe group: {g}\')\n            if g not in self.initial:\n                self.initial[g] = self.previous[g] = self.smoothed[g] = value\n                scores[g] = 0.\n                continue\n            smooth = self.ema * self.smoothed[g] + (1 - self.ema) * value\n            progress = max(0., (self.smoothed[g] - smooth) / max(self.initial[g], 1e-8))\n            scores[g] = progress if self.mode == \'progress\' else smooth / max(self.initial[g], 1e-8)\n            self.previous[g], self.smoothed[g] = self.smoothed[g], smooth\n        if self.mode in (\'uniform\', \'balanced\'):\n            return self.weights\n        # Groups without probes retain a neutral score; coverage is reported.\n        scores = {g: scores.get(g, 0.) for g in self.groups}\n        peak = max(scores.values())\n        exp = {g: math.exp((v - peak) / self.temperature) for g, v in scores.items()}\n        avg = sum(exp.values()) / len(exp)\n        raw = {g: (1 - self.mix) + self.mix * exp[g] / avg for g in self.groups}\n        self.weights = self._bounded(raw)\n        return self.weights\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    temp.replace(path)\n', 'training/train_distill.py': '"""Kaggle pilot: offline-response distillation, with uniform/adaptive group loss.\n\nNo teacher model, generated labels, or benchmark test feedback is used.\nHeavy dependencies are imported only by prepare/train/export commands.\n"""\nimport argparse\nfrom collections import Counter, defaultdict\nfrom importlib.metadata import version\nimport json\nfrom pathlib import Path\nimport random\nimport time\n\nfrom distill_core import (GroupWeights, digest, file_hash, mapped_record, read_records,\n                          split_records, write_json)\n\n\ndef audit(args):\n    fields, groups, examples = Counter(), Counter(), []\n    count = 0\n    rng = random.Random(42)\n    for line, obj in read_records(args.file):\n        count += 1\n        fields.update(obj.keys())\n        groups[str(obj.get(args.group_field, \'<missing>\'))] += 1\n        if len(examples) < args.examples:\n            examples.append({\'line\': line, \'record\': obj})\n        else:\n            index = rng.randrange(count)\n            if index < args.examples:\n                examples[index] = {\'line\': line, \'record\': obj}\n    report = {\'file\': str(Path(args.file).resolve()), \'sha256\': file_hash(args.file),\n              \'records\': count, \'fields\': dict(fields), \'groups\': dict(groups), \'examples\': examples}\n    write_json(args.output, report)\n    print(json.dumps(report, ensure_ascii=False, indent=2))\n\n\ndef prepare(args):\n    if not args.mapping_confirmed:\n        raise ValueError(\'Inspect audit.json first; then explicitly pass --mapping_confirmed with verified field names.\')\n    if args.max_length < 2:\n        raise ValueError(\'max_length must include prompt and response\')\n    out = Path(args.output)\n    if out.exists() and any(out.iterdir()):\n        raise ValueError(\'Prepared output must be empty; choose a new directory to preserve prior manifests.\')\n    from huggingface_hub import HfApi\n    from transformers import AutoTokenizer\n    revision = HfApi().model_info(args.model, revision=args.model_revision).sha\n    tokenizer = AutoTokenizer.from_pretrained(args.model, revision=revision)\n    if tokenizer.eos_token_id is None or not tokenizer.chat_template:\n        raise ValueError(\'Tokenizer requires chat template and EOS\')\n    file_path = Path(args.file)\n    if not file_path.exists():\n        from huggingface_hub import hf_hub_download\n        print(f"Corpus file not found at {args.file}, downloading from Hugging Face hub...")\n        fname = file_path.name if file_path.name.endswith(\'.jsonl\') else "Qwen3-32B_50k.jsonl"\n        downloaded = hf_hub_download(\n            repo_id="zhang-yice/sentiment-distillation-v2",\n            filename=fname,\n            repo_type="dataset",\n            revision=args.corpus_revision\n        )\n        file_path = Path(downloaded)\n        print(f"Corpus downloaded and resolved to: {file_path}")\n    records, rejected, seen = [], [], set()\n    source_sha = file_hash(file_path)\n    fields = args.prompt_fields.split(\',\')\n    for line, obj in read_records(file_path):\n        record_id = f\'{source_sha}:{line}\'\n        try:\n            r = mapped_record(obj, fields, args.response_field, args.group_field, args.source_field)\n            key = digest([r[\'prompt\'], r[\'response\'], r[\'group\']])\n            if key in seen:\n                raise ValueError(\'duplicate prompt/response/group\')\n            seen.add(key)\n            prefix = tokenizer.apply_chat_template(\n                [{\'role\': \'user\', \'content\': r[\'prompt\']}], tokenize=False,\n                add_generation_prompt=True, enable_thinking=False)\n            prompt_ids = tokenizer.encode(prefix, add_special_tokens=False)\n            response_ids = tokenizer.encode(r[\'response\'], add_special_tokens=False)\n            if not response_ids:\n                raise ValueError(\'empty response tokens\')\n            if response_ids[-1] != tokenizer.eos_token_id:\n                response_ids.append(tokenizer.eos_token_id)\n            if len(prompt_ids) + len(response_ids) > args.max_length:\n                raise ValueError(\'overlength: whole record excluded; no response truncation\')\n            r.update(id=record_id, input_ids=prompt_ids + response_ids,\n                     labels=[-100] * len(prompt_ids) + response_ids,\n                     prompt_tokens=len(prompt_ids), response_tokens=len(response_ids))\n            records.append(r)\n        except ValueError as exc:\n            rejected.append({\'id\': record_id, \'reason\': str(exc)})\n    train, probe, holdout, mapping = split_records(\n        records, args.seed, args.dev_fraction, args.max_train, args.probe_per_group,\n        args.holdout_per_group, args.min_group_train, args.max_groups)\n    out.mkdir(parents=True, exist_ok=True)\n    for name, rows in [(\'train\', train), (\'probe\', probe), (\'holdout\', holdout)]:\n        with (out / f\'{name}.jsonl\').open(\'w\', encoding=\'utf-8\') as stream:\n            for r in rows:\n                stream.write(json.dumps(r, ensure_ascii=False) + \'\\n\')\n    tokenizer.save_pretrained(out / \'tokenizer\')\n    clean_config = {k: v for k, v in vars(args).items() if k != \'func\'}\n    clean_config[\'file\'] = str(file_path)\n    manifest = {\'config\': clean_config, \'model\': args.model, \'model_revision\': revision,\n                \'source_sha256\': source_sha, \'corpus_revision\': args.corpus_revision,\n                \'tokenizer_template_sha256\': digest(tokenizer.chat_template), \'group_mapping\': mapping,\n                \'tokenizer_hashes\': {p.name: file_hash(p) for p in sorted((out / \'tokenizer\').iterdir()) if p.is_file()},\n                \'eligible\': len(records), \'rejected\': len(rejected),\n                \'counts\': {name: len(rows) for name, rows in [(\'train\', train), (\'probe\', probe), (\'holdout\', holdout)]},\n                \'train_groups\': dict(Counter(r[\'train_group\'] for r in train)),\n                \'probe_groups\': dict(Counter(r[\'train_group\'] for r in probe)),\n                \'holdout_groups\': dict(Counter(r[\'train_group\'] for r in holdout)),\n                \'split_hashes\': {name: file_hash(out / f\'{name}.jsonl\') for name in (\'train\', \'probe\', \'holdout\')},\n                \'train_input_tokens\': sum(len(r[\'input_ids\']) for r in train),\n                \'train_response_tokens\': sum(r[\'response_tokens\'] for r in train)}\n    write_json(out / \'rejected.json\', rejected)\n    write_json(out / \'manifest.json\', manifest)\n    write_json(out / \'preview.json\', [{k: r[k] for k in (\'id\', \'prompt\', \'response\', \'group\', \'train_group\')}\n                                     for r in train[:12]])\n    print(json.dumps(manifest, indent=2, ensure_ascii=False))\n    print(\'Inspect preview.json, exclusions and group coverage before training.\')\n\n\ndef per_example_loss(logits, labels):\n    """Response-only mean CE, chunked to avoid fp32 [batch, sequence, vocab]."""\n    import torch\n    import torch.nn.functional as F\n    result = []\n    for row, target in zip(logits, labels):\n        target = target[1:]\n        positions = torch.where(target != -100)[0]\n        if not positions.numel():\n            raise ValueError(\'Sample has no response targets\')\n        total = row.new_zeros((), dtype=torch.float32)\n        for indices in positions.split(128):\n            total = total + F.cross_entropy(row[indices].float(), target[indices], reduction=\'sum\')\n        result.append(total / positions.numel())\n    return torch.stack(result)\n\n\ndef _disable_torchao():\n    """Kaggle pre-installs torchao==0.10.0 which triggers PEFT >= 0.16.0 check crash."""\n    import sys\n    try:\n        import peft.import_utils\n        peft.import_utils.is_torchao_available = lambda: False\n    except Exception:\n        pass\n    try:\n        import peft.tuners.lora.torchao\n        peft.tuners.lora.torchao.is_torchao_available = lambda: False\n        peft.tuners.lora.torchao.dispatch_torchao = lambda *args, **kwargs: None\n    except Exception:\n        pass\n    try:\n        import peft.tuners.lora.layer\n        peft.tuners.lora.layer.is_torchao_available = lambda: False\n    except Exception:\n        pass\n    try:\n        import peft.tuners.lora.model\n        peft.tuners.lora.model.is_torchao_available = lambda: False\n    except Exception:\n        pass\n    for mod in list(sys.modules.values()):\n        if hasattr(mod, \'is_torchao_available\'):\n            try:\n                mod.is_torchao_available = lambda: False\n            except Exception:\n                pass\n\n\ndef train(args):\n    import os\n    os.environ.setdefault(\'CUBLAS_WORKSPACE_CONFIG\', \':4096:8\')\n    import numpy as np\n    import torch\n    _disable_torchao()\n    from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training\n    _disable_torchao()\n    from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,\n                              get_cosine_schedule_with_warmup, set_seed)\n    if not torch.cuda.is_available():\n        raise ValueError(\'Train on Kaggle/Colab CUDA GPU; CPU training disabled\')\n    if int(os.environ.get(\'WORLD_SIZE\', \'1\')) != 1:\n        raise ValueError(\'This pilot supports one process/GPU; do not launch with torchrun\')\n    if min(args.steps, args.batch_size, args.grad_accum, args.probe_every, args.save_every, args.log_every, args.rank) < 1:\n        raise ValueError(\'steps, batch, accumulation and intervals must be positive\')\n    if args.learning_rate <= 0 or not 0 <= args.warmup_ratio <= 1:\n        raise ValueError(\'Invalid learning rate/warmup\')\n    prepared, out = Path(args.prepared), Path(args.output)\n    manifest = json.loads((prepared / \'manifest.json\').read_text())\n    for split, expected in manifest[\'split_hashes\'].items():\n        if file_hash(prepared / f\'{split}.jsonl\') != expected:\n            raise ValueError(f\'Prepared split changed: {split}\')\n    for name, expected in manifest[\'tokenizer_hashes\'].items():\n        if file_hash(prepared / \'tokenizer\' / name) != expected:\n            raise ValueError(f\'Prepared tokenizer changed: {name}\')\n    data = {name: [r for _, r in read_records(prepared / f\'{name}.jsonl\')]\n            for name in (\'train\', \'probe\', \'holdout\')}\n    tokenizer = AutoTokenizer.from_pretrained(prepared / \'tokenizer\')\n    tokenizer.pad_token = tokenizer.eos_token\n    set_seed(args.seed)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.benchmark = False\n    bf16 = torch.cuda.is_bf16_supported()\n    dtype = torch.bfloat16 if bf16 else torch.float16\n    versions = {p: version(p) for p in (\'torch\', \'transformers\', \'peft\', \'accelerate\', \'numpy\')}\n    if args.four_bit:\n        versions[\'bitsandbytes\'] = version(\'bitsandbytes\')\n    config = {k: v for k, v in vars(args).items() if k not in (\'output\', \'resume\', \'func\')}\n    identity = {\'config\': config, \'manifest_sha256\': file_hash(prepared / \'manifest.json\'),\n                \'source_sha256\': digest([file_hash(Path(__file__)), file_hash(Path(__file__).with_name(\'distill_core.py\'))]),\n                \'versions\': versions, \'gpu\': torch.cuda.get_device_name(0), \'dtype\': str(dtype)}\n    if out.exists() and any(out.iterdir()) and not args.resume:\n        raise ValueError(\'Output is not empty. Use --resume or choose a new run directory.\')\n    checkpoint = None\n    if args.resume:\n        checkpoint = Path(args.resume).resolve()\n        if checkpoint.parent != out.resolve():\n            raise ValueError(\'Resume in the original run output directory\')\n        if json.loads((checkpoint / \'identity.json\').read_text()) != identity:\n            raise ValueError(\'Resume configuration/data/source/environment differs\')\n    out.mkdir(parents=True, exist_ok=True)\n    write_json(out / \'identity.json\', identity)\n    options = {\'revision\': manifest[\'model_revision\'], \'torch_dtype\': dtype,\n               \'device_map\': {\'\': 0}, \'attn_implementation\': \'sdpa\'}\n    if args.four_bit:\n        options[\'quantization_config\'] = BitsAndBytesConfig(\n            load_in_4bit=True, bnb_4bit_quant_type=\'nf4\', bnb_4bit_use_double_quant=True,\n            bnb_4bit_compute_dtype=dtype)\n    model = AutoModelForCausalLM.from_pretrained(manifest[\'model\'], **options)\n    model.config.use_cache = False\n    if args.four_bit:\n        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)\n    else:\n        model.gradient_checkpointing_enable()\n        model.enable_input_require_grads()\n    if checkpoint:\n        model = PeftModel.from_pretrained(model, checkpoint / \'adapter\', is_trainable=True)\n    else:\n        target_mods = [m.strip() for m in args.target_modules.split(\',\') if m.strip()]\n        model = get_peft_model(model, LoraConfig(\n            task_type=\'CAUSAL_LM\', r=args.rank, lora_alpha=args.rank * 2,\n            lora_dropout=0., target_modules=target_mods))\n    model.print_trainable_parameters()\n    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),\n                                 lr=args.learning_rate, weight_decay=0.01)\n    scheduler = get_cosine_schedule_with_warmup(optimizer, round(args.steps * args.warmup_ratio), args.steps)\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=not bf16)\n    counts = manifest[\'train_groups\']\n    if args.mode != \'uniform\' and len(counts) < 2:\n        raise ValueError(\'Adaptive/group balancing requires at least two train groups; inspect grouping metadata\')\n    controller = GroupWeights(sorted(counts), counts, args.mode, args.mix,\n                              args.temperature, args.ema, args.weight_floor, args.weight_ceiling)\n    state = {\'step\': 0, \'epoch\': 0, \'offset\': 0, \'train_tokens\': 0, \'response_tokens\': 0,\n             \'probe_tokens\': 0, \'holdout_tokens\': 0, \'elapsed_seconds\': 0., \'probe_seconds\': 0.,\n             \'holdout_seconds\': 0., \'optimizer_updates\': 0, \'peak_vram_bytes\': 0}\n    if checkpoint:\n        # Only load a checkpoint produced by this script from your own run.\n        saved = torch.load(checkpoint / \'state.pt\', map_location=\'cpu\', weights_only=False)\n        optimizer.load_state_dict(saved[\'optimizer\'])\n        scheduler.load_state_dict(saved[\'scheduler\'])\n        scaler.load_state_dict(saved[\'scaler\'])\n        controller.__dict__.update(saved[\'controller\'])\n        state = saved[\'state\']\n        random.setstate(saved[\'python_rng\'])\n        np.random.set_state(saved[\'numpy_rng\'])\n        torch.set_rng_state(saved[\'torch_rng\'])\n        torch.cuda.set_rng_state_all(saved[\'cuda_rng\'])\n        if state[\'step\'] >= args.steps:\n            raise ValueError(\'Checkpoint already reached requested steps\')\n        # Remove stale post-checkpoint log entries after a crash, without duplicating steps.\n        log = out / \'metrics.jsonl\'\n        if log.exists():\n            lines = [json.loads(s) for s in log.read_text().splitlines() if s.strip()]\n            log.write_text(\'\'.join(json.dumps(r) + \'\\n\' for r in lines if r[\'step\'] <= state[\'step\']))\n    start = time.monotonic()\n    prior_elapsed = state[\'elapsed_seconds\']\n    torch.cuda.reset_peak_memory_stats()\n\n    def batch(rows):\n        length = max(len(r[\'input_ids\']) for r in rows)\n        ids = [r[\'input_ids\'] + [tokenizer.pad_token_id] * (length - len(r[\'input_ids\'])) for r in rows]\n        labels = [r[\'labels\'] + [-100] * (length - len(r[\'labels\'])) for r in rows]\n        mask = [[1] * len(r[\'input_ids\']) + [0] * (length - len(r[\'input_ids\'])) for r in rows]\n        return {k: torch.tensor(v, device=\'cuda\') for k, v in\n                [(\'input_ids\', ids), (\'attention_mask\', mask), (\'labels\', labels)]}\n\n    def measure(split):\n        torch.cuda.synchronize()\n        begin = time.monotonic()\n        totals, sizes = defaultdict(float), Counter()\n        model.eval()\n        with torch.inference_mode(), torch.autocast(\'cuda\', dtype=dtype):\n            for r in data[split]:\n                b = batch([r])\n                labels = b.pop(\'labels\')\n                output = model(**b, use_cache=False)\n                loss = per_example_loss(output.logits, labels).item()\n                totals[r[\'train_group\']] += loss\n                sizes[r[\'train_group\']] += 1\n                state[f\'{split}_tokens\'] += len(r[\'input_ids\'])\n                del output\n        model.train()\n        torch.cuda.synchronize()\n        state[f\'{split}_seconds\'] += time.monotonic() - begin\n        return {g: totals[g] / sizes[g] for g in totals}\n\n    def log(value):\n        with (out / \'metrics.jsonl\').open(\'a\') as stream:\n            stream.write(json.dumps({\'step\': state[\'step\'], **value}) + \'\\n\')\n        print(json.dumps({\'step\': state[\'step\'], **value}), flush=True)\n\n    def save():\n        torch.cuda.synchronize()\n        state[\'elapsed_seconds\'] = prior_elapsed + time.monotonic() - start\n        state[\'peak_vram_bytes\'] = max(state[\'peak_vram_bytes\'], torch.cuda.max_memory_allocated())\n        target = out / f"checkpoint-{state[\'step\']}"\n        temp = out / f"checkpoint-{state[\'step\']}.tmp"\n        if target.exists() or temp.exists():\n            raise ValueError(f\'Checkpoint path exists: {target}; preserve or move it before resuming\')\n        temp.mkdir()\n        model.save_pretrained(temp / \'adapter\')\n        tokenizer.save_pretrained(temp / \'adapter\')\n        write_json(temp / \'identity.json\', identity)\n        torch.save({\'optimizer\': optimizer.state_dict(), \'scheduler\': scheduler.state_dict(),\n                    \'scaler\': scaler.state_dict(), \'controller\': controller.__dict__, \'state\': state,\n                    \'python_rng\': random.getstate(), \'numpy_rng\': np.random.get_state(),\n                    \'torch_rng\': torch.get_rng_state(), \'cuda_rng\': torch.cuda.get_rng_state_all()}, temp / \'state.pt\')\n        temp.rename(target)\n        write_json(out / \'latest.json\', {\'checkpoint\': str(target.resolve()), \'state\': state})\n\n    if not checkpoint:\n        initial = measure(\'probe\')\n        controller.update(initial)\n        log({\'event\': \'initial_probe\', \'probe_loss\': initial, \'weights\': controller.weights,\n             \'holdout_loss\': measure(\'holdout\')})\n    model.train()\n    order_epoch, order = None, None\n    while state[\'step\'] < args.steps:\n        optimizer.zero_grad(set_to_none=True)\n        micro_losses = []\n        for _ in range(args.grad_accum):\n            rows = []\n            while len(rows) < args.batch_size:\n                if order_epoch != state[\'epoch\']:\n                    order_epoch = state[\'epoch\']\n                    order = list(range(len(data[\'train\'])))\n                    random.Random(args.seed + state[\'epoch\']).shuffle(order)\n                take = min(args.batch_size - len(rows), len(order) - state[\'offset\'])\n                rows.extend(data[\'train\'][i] for i in order[state[\'offset\']:state[\'offset\'] + take])\n                state[\'offset\'] += take\n                if state[\'offset\'] == len(order):\n                    state[\'epoch\'] += 1\n                    state[\'offset\'] = 0\n            b = batch(rows)\n            labels = b.pop(\'labels\')\n            with torch.autocast(\'cuda\', dtype=dtype):\n                output = model(**b, use_cache=False)\n                losses = per_example_loss(output.logits, labels)\n                weights = losses.new_tensor([controller.weights[r[\'train_group\']] for r in rows])\n                loss = (losses * weights).mean() / args.grad_accum\n            if not torch.isfinite(loss):\n                raise FloatingPointError(\'Non-finite loss; stop and lower LR/check data/precision\')\n            scaler.scale(loss).backward()\n            micro_losses.append(losses.detach().mean().item())\n            state[\'train_tokens\'] += sum(len(r[\'input_ids\']) for r in rows)\n            state[\'response_tokens\'] += sum(r[\'response_tokens\'] for r in rows)\n            del output, losses, loss\n        scaler.unscale_(optimizer)\n        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)\n        old_scale = scaler.get_scale()\n        scaler.step(optimizer)\n        scaler.update()\n        if scaler.get_scale() >= old_scale:\n            scheduler.step()\n            state[\'optimizer_updates\'] += 1\n        state[\'step\'] += 1\n        if state[\'step\'] % args.log_every == 0 or state[\'step\'] == args.steps:\n            log({\'event\': \'train\', \'response_ce\': sum(micro_losses) / len(micro_losses),\n                 \'grad_norm\': float(grad_norm), \'learning_rate\': scheduler.get_last_lr()[0],\n                 \'train_tokens\': state[\'train_tokens\'], \'response_tokens\': state[\'response_tokens\'],\n                 \'optimizer_updates\': state[\'optimizer_updates\']})\n        if state[\'step\'] % args.probe_every == 0 or state[\'step\'] == args.steps:\n            losses = measure(\'probe\')\n            if state[\'step\'] >= args.warmup_steps:\n                controller.update(losses)\n            log({\'event\': \'probe\', \'probe_loss\': losses, \'weights\': controller.weights,\n                 \'holdout_loss\': measure(\'holdout\')})\n        if state[\'step\'] % args.save_every == 0 or state[\'step\'] == args.steps:\n            save()\n    model.save_pretrained(out / \'adapter\')\n    tokenizer.save_pretrained(out / \'adapter\')\n    write_json(out / \'summary.json\', {\'mode\': args.mode, \'model\': manifest[\'model\'], \'state\': state,\n                                    \'weights\': controller.weights,\n                                    \'note\': \'Teacher-response CE is not benchmark accuracy/F1. Run SentiBench separately.\'})\n\n\ndef export(args):\n    import torch\n    _disable_torchao()\n    from peft import PeftModel\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    run = Path(args.run)\n    identity = json.loads((run / \'identity.json\').read_text())\n    prepared = Path(identity[\'config\'][\'prepared\'])\n    manifest = json.loads((prepared / \'manifest.json\').read_text())\n    target = Path(args.output)\n    if target.exists() and any(target.iterdir()):\n        raise ValueError(\'Export target must be empty\')\n    model = AutoModelForCausalLM.from_pretrained(manifest[\'model\'], revision=manifest[\'model_revision\'],\n                                               torch_dtype=torch.float32, device_map={\'\': \'cpu\'})\n    model = PeftModel.from_pretrained(model, run / \'adapter\').merge_and_unload()\n    model.save_pretrained(target, safe_serialization=True)\n    AutoTokenizer.from_pretrained(run / \'adapter\').save_pretrained(target)\n    write_json(target / \'distillation_run.json\', identity)\n    print(\'Merged model:\', target)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    commands = parser.add_subparsers(dest=\'command\', required=True)\n    a = commands.add_parser(\'audit\')\n    a.add_argument(\'--file\', required=True)\n    a.add_argument(\'--group_field\', default=\'cluster_id\')\n    a.add_argument(\'--examples\', type=int, default=5)\n    a.add_argument(\'--output\', required=True)\n    a.set_defaults(func=audit)\n    p = commands.add_parser(\'prepare\')\n    p.add_argument(\'--file\', required=True)\n    p.add_argument(\'--output\', required=True)\n    p.add_argument(\'--model\', default=\'Qwen/Qwen3-0.6B\')\n    p.add_argument(\'--model_revision\', default=\'main\')\n    p.add_argument(\'--corpus_revision\', required=True)\n    p.add_argument(\'--prompt_fields\', required=True, help=\'Comma-separated fields in verified order\')\n    p.add_argument(\'--response_field\', required=True)\n    p.add_argument(\'--group_field\', required=True)\n    p.add_argument(\'--source_field\', required=True, help=\'Original input text for leakage-safe grouped splitting\')\n    p.add_argument(\'--mapping_confirmed\', action=\'store_true\')\n    p.add_argument(\'--max_length\', type=int, default=2048)\n    p.add_argument(\'--max_train\', type=int, default=5000, help=\'0 = all eligible training records\')\n    p.add_argument(\'--seed\', type=int, default=42)\n    p.add_argument(\'--dev_fraction\', type=float, default=0.1)\n    p.add_argument(\'--probe_per_group\', type=int, default=4)\n    p.add_argument(\'--holdout_per_group\', type=int, default=4)\n    p.add_argument(\'--min_group_train\', type=int, default=8)\n    p.add_argument(\'--max_groups\', type=int, default=16)\n    p.set_defaults(func=prepare)\n    t = commands.add_parser(\'train\')\n    t.add_argument(\'--prepared\', required=True)\n    t.add_argument(\'--output\', required=True)\n    t.add_argument(\'--mode\', choices=[\'uniform\', \'balanced\', \'loss\', \'progress\'], default=\'uniform\')\n    t.add_argument(\'--steps\', type=int, default=200)\n    t.add_argument(\'--batch_size\', type=int, default=1)\n    t.add_argument(\'--grad_accum\', type=int, default=8)\n    t.add_argument(\'--learning_rate\', type=float, default=1e-4)\n    t.add_argument(\'--rank\', type=int, default=16)\n    t.add_argument(\'--target_modules\', default=\'q_proj,k_proj,v_proj,o_proj\',\n                   help=\'Comma-separated LoRA target module names\')\n    t.add_argument(\'--warmup_ratio\', type=float, default=0.05)\n    t.add_argument(\'--warmup_steps\', type=int, default=25, help=\'Before adaptive updates\')\n    t.add_argument(\'--probe_every\', type=int, default=25)\n    t.add_argument(\'--save_every\', type=int, default=100)\n    t.add_argument(\'--log_every\', type=int, default=10)\n    t.add_argument(\'--mix\', type=float, default=0.5)\n    t.add_argument(\'--temperature\', type=float, default=0.05)\n    t.add_argument(\'--ema\', type=float, default=0.5)\n    t.add_argument(\'--weight_floor\', type=float, default=0.5)\n    t.add_argument(\'--weight_ceiling\', type=float, default=2.)\n    t.add_argument(\'--seed\', type=int, default=42)\n    t.add_argument(\'--four_bit\', action=\'store_true\')\n    t.add_argument(\'--resume\', help=\'Own checkpoint directory; restores optimizer/scaler/RNG/controller/cursor\')\n    t.set_defaults(func=train)\n    e = commands.add_parser(\'export\')\n    e.add_argument(\'--run\', required=True)\n    e.add_argument(\'--output\', required=True)\n    e.set_defaults(func=export)\n    args = parser.parse_args()\n    args.func(args)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'requirements-train-kaggle.txt': '# Reuse the platform CUDA PyTorch. Notebook blocks torch/nvidia/triton changes.\n# Known API target; live dependency/import/GPU checks run on Kaggle, not locally.\ntransformers==4.51.3\ntokenizers==0.21.4\nhuggingface-hub==0.30.2\npeft==0.15.2\naccelerate==1.6.0\nsafetensors>=0.5.3,<0.6\n# Optional QLoRA only: bitsandbytes==0.46.1 (not required for ordinary LoRA).\n', 'kaggle_evaluate.py': '"""Kaggle evaluation suite: evaluate and compare Base Model (Before) vs Distilled Model (After LoRA).\n\nRuns evaluation in separate subprocesses to prevent CUDA memory fragmentation and OOM.\nGenerates side-by-side comparison tables and saves evaluation_comparison.json / .md.\n"""\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\n\ndef run_single_eval(script_name, model, adapter, dataset, k_shot, batch_size,\n                    max_seq_length, max_samples, output_dir, seed):\n    output_dir = Path(output_dir).resolve()\n    output_dir.mkdir(parents=True, exist_ok=True)\n    subname = \'eval_finetuned\' if adapter else \'eval_base\'\n\n    cmd = [\n        sys.executable,\n        str(Path(__file__).parent / script_name),\n        \'--dataset\', dataset,\n        \'--subname\', subname,\n        \'--seed\', str(seed),\n        \'--model_name_or_path\', model,\n        \'--k_shot\', str(k_shot),\n        \'--eval_batch_size\', str(batch_size),\n        \'--max_seq_length\', str(max_seq_length),\n        \'--output_dir\', str(output_dir),\n    ]\n    if adapter:\n        cmd.extend([\'--adapter\', str(adapter)])\n    if max_samples > 0:\n        cmd.extend([\'--max_samples\', str(max_samples)])\n\n    print(f"\\n{\'=\'*70}")\n    target_desc = f"{model} + Adapter ({adapter})" if adapter else f"Base Model ({model})"\n    print(f"Running evaluation: {dataset} | {target_desc}")\n    print(f"Command: {\' \'.join(cmd)}")\n    print(f"{\'=\'*70}\\n")\n\n    start_time = time.time()\n    res = subprocess.run(cmd, check=True)\n    elapsed = time.time() - start_time\n\n    # Read latest result from result.txt\n    result_file = output_dir / \'result.txt\'\n    if not result_file.exists():\n        raise RuntimeError(f"Expected result file not found: {result_file}")\n\n    last_record = None\n    with open(result_file, \'r\', encoding=\'utf-8-sig\') as f:\n        for line in f:\n            if line.strip():\n                try:\n                    last_record = json.loads(line)\n                except Exception:\n                    pass\n\n    if not last_record:\n        raise RuntimeError(f"Could not parse valid result record from {result_file}")\n\n    metrics = last_record[1]\n    return {\n        \'f1\': metrics.get(\'f1\', 0.0),\n        \'accuracy\': metrics.get(\'acc\', 0.0),\n        \'elapsed_seconds\': round(elapsed, 2),\n        \'experiment_name\': last_record[0]\n    }\n\n\ndef format_delta(base_val, ft_val):\n    delta = ft_val - base_val\n    sign = \'+\' if delta > 0 else \'\'\n    icon = \'▲\' if delta > 0.0005 else (\'▼\' if delta < -0.0005 else \'=\')\n    return f"{ft_val*100:.2f}% ({sign}{delta*100:.2f}% {icon})"\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--model\', default=\'Qwen/Qwen2.5-0.5B-Instruct\', help=\'Base model HF ID or path\')\n    parser.add_argument(\'--adapter\', default=None, help=\'Path to trained PEFT LoRA adapter directory\')\n    parser.add_argument(\'--datasets\', default=\'sc/imdb,sc/sst2\', help=\'Comma-separated datasets to evaluate\')\n    parser.add_argument(\'--k_shot\', type=int, default=4, help=\'Few-shot demonstrations\')\n    parser.add_argument(\'--eval_batch_size\', type=int, default=4, help=\'Evaluation batch size\')\n    parser.add_argument(\'--max_seq_length\', type=int, default=2048, help=\'Context window cutoff\')\n    parser.add_argument(\'--max_samples\', type=int, default=0, help=\'Max test examples (0 for all)\')\n    parser.add_argument(\'--output_dir\', default=\'./output/evaluation_comparison\', help=\'Output report directory\')\n    parser.add_argument(\'--seed\', type=int, default=42)\n    parser.add_argument(\'--skip_base\', action=\'store_true\', help=\'Skip base model evaluation if already recorded\')\n    args = parser.parse_args()\n\n    out_dir = Path(args.output_dir).resolve()\n    out_dir.mkdir(parents=True, exist_ok=True)\n    datasets = [d.strip() for d in args.datasets.split(\',\') if d.strip()]\n\n    report = {\n        \'timestamp\': time.strftime(\'%Y-%m-%d %H:%M:%S\'),\n        \'base_model\': args.model,\n        \'adapter\': args.adapter,\n        \'k_shot\': args.k_shot,\n        \'max_samples\': args.max_samples,\n        \'results\': {}\n    }\n\n    base_dir = out_dir / \'raw_base\'\n    ft_dir = out_dir / \'raw_finetuned\'\n\n    for ds in datasets:\n        script = \'fsa_kaggle.py\' if ds.startswith(\'absa/\') else \'icl_multifaced_kaggle.py\'\n        report[\'results\'][ds] = {}\n\n        # 1. Base Model evaluation\n        base_res = None\n        if not args.skip_base:\n            base_res = run_single_eval(\n                script_name=script,\n                model=args.model,\n                adapter=None,\n                dataset=ds,\n                k_shot=args.k_shot,\n                batch_size=args.eval_batch_size,\n                max_seq_length=args.max_seq_length,\n                max_samples=args.max_samples,\n                output_dir=base_dir,\n                seed=args.seed\n            )\n            report[\'results\'][ds][\'base\'] = base_res\n\n        # 2. Fine-tuned Model evaluation (if adapter specified)\n        if args.adapter:\n            ft_res = run_single_eval(\n                script_name=script,\n                model=args.model,\n                adapter=args.adapter,\n                dataset=ds,\n                k_shot=args.k_shot,\n                batch_size=args.eval_batch_size,\n                max_seq_length=args.max_seq_length,\n                max_samples=args.max_samples,\n                output_dir=ft_dir,\n                seed=args.seed\n            )\n            report[\'results\'][ds][\'finetuned\'] = ft_res\n\n            if base_res:\n                report[\'results\'][ds][\'delta_f1\'] = round(ft_res[\'f1\'] - base_res[\'f1\'], 4)\n                report[\'results\'][ds][\'delta_accuracy\'] = round(ft_res[\'accuracy\'] - base_res[\'accuracy\'], 4)\n\n    # 3. Print & Save Comparison Table\n    print("\\n" + "="*84)\n    print("           SENTIMENT DISTILLATION: BEFORE vs AFTER EVALUATION REPORT")\n    print("="*84)\n    print(f"Base Model:         {args.model}")\n    print(f"Adapter Checkpoint: {args.adapter or \'<None>\'}")\n    print(f"Few-shot (k_shot):  {args.k_shot}")\n    print(f"Test samples:       {\'All\' if args.max_samples == 0 else args.max_samples}")\n    print("-"*84)\n    header = f"{\'Dataset\':<20} | {\'Metric\':<10} | {\'Base Model (Before)\':<20} | {\'Distilled (After)\':<26}"\n    print(header)\n    print("-"*84)\n\n    lines = []\n    lines.append(f"# Báo cáo Đối chứng Đánh giá Trước vs Sau Chưng cất (Sentiment Distillation)\\n")\n    lines.append(f"- **Mô hình Gốc (Base)**: `{args.model}`")\n    lines.append(f"- **LoRA Adapter**: `{args.adapter or \'N/A\'}`")\n    lines.append(f"- **Số mẫu Demo (k-shot)**: `{args.k_shot}`")\n    lines.append(f"- **Thời gian đánh giá**: `{report[\'timestamp\']}`\\n")\n    lines.append(f"| Tác vụ (Dataset) | Chỉ số | Mô hình Gốc (Trước) | Sau Finetuning (Adapter) | Mức độ cải thiện (Delta) |")\n    lines.append(f"|---|---|---|---|---|")\n\n    for ds, data in report[\'results\'].items():\n        base = data.get(\'base\')\n        ft = data.get(\'finetuned\')\n\n        if base and ft:\n            f1_str = format_delta(base[\'f1\'], ft[\'f1\'])\n            acc_str = format_delta(base[\'accuracy\'], ft[\'accuracy\'])\n            print(f"{ds:<20} | {\'Macro F1\':<10} | {base[\'f1\']*100:>6.2f}%              | {f1_str:<26}")\n            print(f"{ds:<20} | {\'Accuracy\':<10} | {base[\'accuracy\']*100:>6.2f}%              | {acc_str:<26}")\n            print("-"*84)\n            d_f1 = ft[\'f1\'] - base[\'f1\']\n            d_acc = ft[\'accuracy\'] - base[\'accuracy\']\n            lines.append(f"| `{ds}` | Macro F1 | {base[\'f1\']*100:.2f}% | {ft[\'f1\']*100:.2f}% | **{\'+\' if d_f1>=0 else \'\'}{d_f1*100:.2f}%** |")\n            lines.append(f"| `{ds}` | Accuracy | {base[\'accuracy\']*100:.2f}% | {ft[\'accuracy\']*100:.2f}% | **{\'+\' if d_acc>=0 else \'\'}{d_acc*100:.2f}%** |")\n        elif base:\n            print(f"{ds:<20} | {\'Macro F1\':<10} | {base[\'f1\']*100:>6.2f}%              | {\'N/A\':<26}")\n            print(f"{ds:<20} | {\'Accuracy\':<10} | {base[\'accuracy\']*100:>6.2f}%              | {\'N/A\':<26}")\n            print("-"*84)\n            lines.append(f"| `{ds}` | Macro F1 | {base[\'f1\']*100:.2f}% | N/A | N/A |")\n            lines.append(f"| `{ds}` | Accuracy | {base[\'accuracy\']*100:.2f}% | N/A | N/A |")\n\n    print("="*84 + "\\n")\n\n    # Save JSON and Markdown\n    json_path = out_dir / \'evaluation_comparison.json\'\n    md_path = out_dir / \'evaluation_comparison.md\'\n    json_path.write_text(json.dumps(report, indent=2, ensure_ascii=False))\n    md_path.write_text(\'\\n\'.join(lines), encoding=\'utf-8\')\n    print(f"Detailed comparison saved to:")\n    print(f"  JSON: {json_path}")\n    print(f"  Markdown: {md_path}\\n")\n\n\nif __name__ == \'__main__\':\n    main()\n', 'icl_multifaced_kaggle.py': 'from notebook_runtime import (add_runtime_arguments, prepare_runtime, load_model,\n    select_examples, configure_tokenizer, limit_prompts, run_metadata, PredictionCache,\n    save_result_kaggle as save_result)\nfrom tqdm import tqdm\nimport argparse\nfrom utils import save_line_json, load_line_json\nimport random\nfrom collections import Counter\nfrom torch.utils.data import Dataset, DataLoader\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\nimport torch\nfrom utils import load_json,save_json\nfrom sklearn.metrics import f1_score, accuracy_score, classification_report\nimport datetime\nimport time\nimport os\n\n\nid_to_label = {\n    \'multifaced/tweeteval\': [\' anger\', \' joy\', \' sadness\', \' optimism\'],\n    \'multifaced/intimacy\': [\' not intimate\', \' moderately intimate\', \' highly intimate\'],\n    \'multifaced/offenseval\':[\' non-offensive\', \' offensive\'],\n    \'multifaced/irony18\': [\' non-irony\', \' irony\'],\n    \'multifaced/pstance\': [\' against\', \' favor\'],\n    \'sc/imdb\': [\' negative\', \' positive\'],\n    \'sc/sst2\': [\' negative\', \' positive\'],\n    \'sc/yelp2\': [\' negative\', \' positive\'],\n    \'sc/twitter\': [\' negative\', \' positive\', \' neutral\'],\n}\n\n\ndef get_label_list(tokenizer, dataset):\n    ids = [tokenizer(label, add_special_tokens=False)[\'input_ids\'][0]\n           for label in id_to_label[dataset]]\n    if len(set(ids)) != len(ids):\n        raise ValueError(\'Label first tokens collide for this tokenizer; use a compatible model or implement full-label scoring.\')\n    return ids\n\n\ndef cal_metric(results, y_true, y_pred, seed, subname, model_name_or_path, dataset, output_dir, k_shot, metadata=None):\n    model_name = model_name_or_path.split(\'/\')[-1]\n    print(model_name)\n    current_time = time.strftime(\'%Y_%m_%d_%H_%M_%S\')\n    file_name = f\'{subname}_{model_name}_{dataset}_{seed}_numOfshot{k_shot}_{current_time}\'\n    output_path = os.path.join(output_dir, file_name + \'.json\')\n    print(output_path)\n\n    print(Counter(y_true))\n    print(Counter(y_pred))\n\n    f1 = f1_score(y_true, y_pred, average=\'macro\')\n    accuracy = accuracy_score(y_true, y_pred)\n    print(f\'f1: {f1:.4f} | accuracy: {accuracy:.4f}\\n\')\n\n\n    save_result(output_dir, file_name, f1, accuracy, metadata)\n    save_json(results, output_path)\n\n\nclass DataCollator:\n    def __init__(self, tokenizer, max_seq_length, mode):\n        self.tokenizer = tokenizer\n        self.max_seq_length = max_seq_length\n        self.mode = mode\n\n    def tok(self, text, max_seq_length):\n        kwargs = {\n            \'text\': text,\n            \'return_tensors\': \'pt\',\n        }\n\n        if max_seq_length in (-1, \'longest\'):\n            kwargs[\'padding\'] = True\n\n        else:\n            kwargs[\'max_length\'] = max_seq_length\n            kwargs[\'padding\'] = True\n            kwargs[\'truncation\'] = True\n\n        batch_encodings = self.tokenizer(**kwargs)\n        return batch_encodings\n\n    def __call__(self, examples):\n\n        texts = [example[\'prompts\'] for example in examples]\n        batch_encodings = self.tok(texts, self.max_seq_length)\n\n        input_ids = batch_encodings[\'input_ids\']\n        attention_mask = batch_encodings[\'attention_mask\']\n\n        return {\n            \'input_ids\': input_ids,\n            \'attention_mask\': attention_mask,\n            \'examples\': examples\n        }\n\ndef get_dataloader(test_examples, mode, batch_size, shuffle, tokenizer):\n    dataloader = DataLoader(\n        dataset=test_examples,\n        batch_size=batch_size,\n        shuffle=shuffle,\n        pin_memory=True,\n        num_workers=0,\n        collate_fn=DataCollator(\n            tokenizer=tokenizer,\n            max_seq_length=max_seq_length,\n            mode=mode,\n        )\n    )\n\n    print(\'dataloader-\' + mode, len(dataloader))\n    return dataloader\n\n\ndef test_dataloader(test_examples, eval_batch_size,tokenizer):\n    return get_dataloader(test_examples, "test", eval_batch_size, shuffle=False, tokenizer=tokenizer)\n\n\ndef construct_prompt(prompt, test_examples, demo_pool, k_shot):\n\n    for example in test_examples:\n        text = example[\'sentence\']\n        demos = random.sample(demo_pool, k=k_shot)\n        prompts = [prompt]\n        prompts.append(\'\')\n        for demo in demos:\n            prompts.append(f"Sentence: {demo[\'sentence\']}\\nLabel: {demo[\'label\'].lower()}\\n")\n        prompts.append(f"Sentence: {example[\'sentence\']}\\nLabel:")\n        example[\'prompts\'] = \'\\n\'.join(prompts)\n\n\n\n    return test_examples\n\n\ndef construct_prompt_pstance(prompt, test_examples, demo_pool, k_shot):\n\n    for example in test_examples:\n        text = example[\'sentence\']\n        target = example[\'target\']\n        demos = random.sample(demo_pool, k=k_shot)\n        prompts = [prompt % target]\n        prompts.append(\'\')\n        for demo in demos:\n            prompts.append(f"Sentence: {demo[\'sentence\']}\\nLabel: {demo[\'label\'].lower()}(opinion towards \'{demo[\'target\']}\')\\n")\n        prompts.append(f"Sentence: {example[\'sentence\']}\\nLabel:")\n        example[\'prompts\'] = \'\\n\'.join(prompts)\n\n\n\n    return test_examples\n\n\n\ndef load_dataset(data_path, demo_split=\'train\'):\n    test_file_name = os.path.join(data_path, \'test.json\')\n    test_examples = load_json(test_file_name)\n\n    demo_pool_file_name = os.path.join(data_path, demo_split + \'.json\')\n    demo_examples = load_json(demo_pool_file_name)\n\n    raw_datasets = {\n        \'test\': test_examples,\n        \'demo\': demo_examples\n    }\n\n    print(\'-----------data statistic-------------\')\n    for mode in (\'test\',\'demo\'):\n        num_samples = len(raw_datasets[mode])\n        print(f\'{mode.upper():<5} | Samples: {num_samples:<5}\')\n    print(\'--------------------------------------\')\n\n\n\n    return raw_datasets\n\n\n\nimport sys\nimport argparse\nimport datetime\nif __name__ == \'__main__\':\n    print(\'hello\')\n    parser = argparse.ArgumentParser()\n\n    parser.add_argument(\'--dataset\', required=True, )\n    parser.add_argument(\'--subname\', required=True, )\n    parser.add_argument(\'--seed\', required=True, type=int)\n    parser.add_argument(\'--model_name_or_path\', default=None )\n    parser.add_argument(\'--model_version\', default=None)\n    parser.add_argument(\'--max_seq_length\', type=int, default=2048)\n    parser.add_argument(\'--data_prop\', type=float, default=1)\n    parser.add_argument(\'--eval_batch_size\', type=int, default=1)\n    parser.add_argument(\'--data_dir\', default=os.path.join(os.path.dirname(__file__), \'datasets\'))\n    parser.add_argument(\'--output_dir\', default=\'./output\')\n    parser.add_argument(\'--k_shot\', type=int, default=4)\n\n    add_runtime_arguments(parser)\n    args = parser.parse_args()\n    prepare_runtime(args)\n\n    dataset = args.dataset\n    subname = args.subname\n    seed = args.seed\n    model_version = args.model_version\n    model_name_or_path = args.model_name_or_path\n    max_seq_length = args.max_seq_length\n    data_prop = args.data_prop\n    eval_batch_size = args.eval_batch_size\n    data_dir = args.data_dir\n    output_dir = args.output_dir\n    k_shot = args.k_shot\n\n    data_path = os.path.join(data_dir, dataset)\n    torch.manual_seed(seed)\n    random.seed(seed)\n\n    torch.cuda.manual_seed(seed)\n\n\n    model_name_or_path = args.model_name_or_path\n\n    print(\'-\'*20+\'Experiment Setup\'+\'-\'*20)\n    print(\'data_path: \', data_path)\n    print(f\' subname: {subname}\\n dataset: {dataset}\\n seed:{seed}\\n model_name_or_path:{model_name_or_path}\\n data_path:{data_path}\\n output_dir:{output_dir}\\n eval_batch_size:{eval_batch_size}\\n data_prop:{data_prop}\\n max_seq_length:{max_seq_length}\\n numbers_of_shot:{k_shot}\')\n    print(\'-\' * 50)\n\n    prompts = {\n        \'multifaced/irony18\':\n            """Please perform Irony Detection task. Given the sentence, assign a sentiment label from [\'irony\', \'non-irony\']. Return label only without any other text.""",\n        \'multifaced/offenseval\':\n            """Please perform Offensive Detection task. Given the sentence, assign a sentiment label from [\'non-offensive\', \'offensive\']. Return label only without any other text.""",\n        \'multifaced/intimacy\':\n            """Please perform Intimacy Detection task. Given the sentence, assign an intimacy label from [\'not intimate\', \'moderately intimate\', \'highly intimate\']. Return label only without any other text.""",\n        \'multifaced/tweeteval\':\n            """Please perform Emotion Detection task. Given the sentence, assign a emotion label from [\'anger\', \'joy\', \'sadness\', \'optimism\']. Return the label only without any other text.""",\n        \'multifaced/pstance\':\n            """Please perform Stance Detection task. Given the sentence, assign a sentiment label expressed by the author towards "%s" from [\'against\', \'favor\']. Return label only without any other text.""",\n        \'sc/imdb\':\n            """Please perform Sentiment Analysis task. Given the sentence, assign a sentiment polarity label from [\'negative\', \'positive\']. Return label only without any other text.""",\n        \'sc/sst2\':\n            """Please perform Sentiment Analysis task. Given the sentence, assign a sentiment polarity label from [\'negative\', \'positive\']. Return label only without any other text.""",\n        \'sc/yelp2\':\n            """Please perform Sentiment Analysis task. Given the sentence, assign a sentiment polarity label from [\'negative\', \'positive\']. Return label only without any other text.""",\n        \'sc/twitter\':\n            """Please perform Sentiment Analysis task. Given the sentence, assign a sentiment polarity label from [\'negative\', \'positive\', \'neutral\']. Return label only without any other text.""",\n    }\n\n\n\n\n    raw_datasets = select_examples(load_dataset(data_path, args.demo_split), args)\n\n    if dataset == \'multifaced/pstance\':\n        test_examples = construct_prompt_pstance(prompts[dataset], raw_datasets[\'test\'], raw_datasets[\'demo\'], k_shot)\n    else:\n        test_examples = construct_prompt(prompts[dataset], raw_datasets[\'test\'], raw_datasets[\'demo\'], k_shot)\n\n    for i in range(1):\n        print(test_examples[i][\'prompts\'])\n        print(\'-\'*100)\n\n    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, padding_side=\'left\')\n    tokenizer.pad_token = tokenizer.eos_token\n\n\n\n    model = load_model(args)\n    configure_tokenizer(tokenizer, model, args, 1)\n    max_seq_length = args.max_seq_length\n    truncated = limit_prompts(test_examples, tokenizer, max_seq_length)\n    metadata = run_metadata(args, raw_datasets, truncated)\n    cache = PredictionCache(args.resume_file, metadata)\n    pending = [example for example in test_examples if cache.get(example) is None]\n    dataloader = test_dataloader(pending, eval_batch_size, tokenizer)\n\n\n\n\n\n    model.eval()\n\n    label_ids = get_label_list(tokenizer, dataset)\n    results = []\n    with torch.inference_mode():\n        for batch in tqdm(dataloader):\n            # Same first-token decision rule as the original evaluator; no KV cache.\n            output = model(\n                input_ids=batch[\'input_ids\'].to(model.device),\n                attention_mask=batch[\'attention_mask\'].to(model.device),\n                use_cache=False,\n            )\n            scores = output.logits[:, -1, label_ids].float().cpu()\n            for example, score in zip(batch[\'examples\'], scores):\n                cache.put(example, {\'prediction\': id_to_label[dataset][score.argmax().item()].strip(),\n                                    \'prediction_score\': score.tolist()})\n            cache.flush()\n            del output, scores\n\n    for example in test_examples:\n        result = dict(example)\n        result.update(cache.get(example))\n        results.append(result)\n    labels = [example[\'label\'].lower() for example in results]\n    predictions = [example[\'prediction\'] for example in results]\n    cal_metric(results, labels, predictions, seed, subname, model_name_or_path,\n               dataset, output_dir, k_shot, metadata)\n', 'fsa_kaggle.py': 'from notebook_runtime import (add_runtime_arguments, prepare_runtime, load_model,\n    select_examples, configure_tokenizer, limit_prompts, run_metadata, PredictionCache,\n    save_result_kaggle as save_result)\nfrom tqdm import tqdm\nimport argparse\nfrom utils import save_line_json, load_line_json\nimport random\nfrom collections import Counter\nfrom torch.utils.data import Dataset, DataLoader\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\nimport torch\nfrom utils import load_json,save_json\nfrom sklearn.metrics import f1_score, accuracy_score, classification_report\nimport datetime\nimport time\nimport os\n\nfrom parse_utils.asqp import Result_asqp\nfrom parse_utils.ssa import Result_ssa\nfrom parse_utils.coqe import Result_coqe\nfrom parse_utils.atsa import Result_atsa\nfrom parse_utils.acsa import Result_acsa\n\n\ndef ifnt0(x):\n    """\n    Return 0 if the input is None, otherwise return the input.\n    :param x: The input value that may be None.\n    :return: 0 if x is None, x otherwise.\n    """\n    return 0 if x is None else x\n\nclass FineTuneDataset(Dataset):\n    def __init__(self, dataset_name, dataset, demo_pool ,k_shot, model_name_or_path, mode, max_length):\n        self.model_name_or_path = model_name_or_path\n        self.demo_pool = demo_pool\n        self.k_shot = k_shot\n        self.dataset_name = dataset_name\n        self.dataset_dict = {dataset_name: dataset}\n        self.max_length = max_length\n        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name_or_path, use_fast=True, padding_side=\'left\')\n        self.tokenizer.pad_token = self.tokenizer.eos_token\n        self.mode = mode\n        self.prompts = {\n        \'absa/asqp_rest16\':"""Please perform Aspect Sentiment Quad Prediction task. Given the sentence, extract all (aspect term, aspect category, opinion, sentiment polarity) quadruples.\n\n1. Aspect category should be selected from [\'ambience general\', \'drinks prices\',\'drinks quality\', \'drinks style_options\', \'food general\', \'food prices\', \'food quality\', \'food style_options\', \'location general\', \'restaurant general\', \'restaurant miscellaneous\', \'restaurant pcrices\', \'service general\'].\n\n2. Sentiment polarity should be selected from [\'negative\', \'neutral\', \'positive\'].\n\n3. If there is no aspect term, use \'NULL\' as the aspect term. Only aspect term can be \'NULL\', aspect category, opinion and sentiment polarity CANNOT be \'NULL\'.\n\n4. Please return python list only, without any other comments or texts.""",\n        \'absa/asqp_laptop\':"""Please perform Aspect Sentiment Quad Prediction task. Given the sentence, extract all (aspect term, aspect category, opinion, sentiment polarity) quadruples.\n\n1. Aspect category should be selected from [\'battery miscellaneous\', \'battery operation_performance\', \'battery quality\', \'company general\', \'cpu design_features\', \'cpu miscellaneous\', \'cpu operation_performance\', \'cpu quality\', \'display design_features\', \'display general\', \'display operation_performance\', \'display quality\', \'display usability\', \'fans_cooling design_features\', \'fans_cooling operation_performance\', \'fans_cooling quality\', \'graphics design_features\', \'graphics general\', \'graphics miscellaneous\', \'graphics quality\', \'hard_disc design_features\', \'hard_disc quality\', \'hardware general\', \'hardware operation_performance\', \'hardware quality\', \'keyboard design_features\', \'keyboard general\', \'keyboard operation_performance\', \'keyboard quality\', \'keyboard usability\', \'laptop connectivity\', \'laptop design_features\', \'laptop general\', \'laptop miscellaneous\', \'laptop operation_performance\', \'laptop portability\', \'laptop price\', \'laptop quality\', \'laptop usability\', \'memory design_features\', \'motherboard quality\', \'mouse design_features\', \'mouse general\', \'mouse operation_performance\', \'mouse quality\', \'mouse usability\', \'multimedia_devices design_features\', \'multimedia_devices general\', \'multimedia_devices miscellaneous\', \'multimedia_devices operation_performance\', \'multimedia_devices quality\', \'multimedia_devices usability\', \'optical_drives operation_performance\', \'optical_drives quality\', \'os design_features\', \'os general\', \'os miscellaneous\', \'os operation_performance\', \'os quality\', \'os usability\', \'ports design_features\', \'ports operation_performance\', \'ports quality\', \'power_supply design_features\', \'power_supply miscellaneous\', \'power_supply operation_performance\', \'power_supply quality\', \'shipping price\', \'shipping quality\', \'software design_features\', \'software general\', \'software miscellaneous\', \'software operation_performance\', \'software price\', \'software quality\', \'software usability\', \'support miscellaneous\', \'support price\', \'support quality\', \'warranty general\', \'warranty price\'].\n\n2. Sentiment polarity should be selected from [\'negative\', \'neutral\', \'positive\'].\n\n3. If there is no aspect term, use \'NULL\' as the aspect term. Only aspect term can be \'NULL\', aspect category, opinion and sentiment polarity CANNOT be \'NULL\'.\n\n4. Please return python list only, without any other comments or texts.""",\n\n            \'absa/opener\':"""Please perform the Structured Sentiment Analysis task. Given a sentence, extract all opinion tuples in the format (holder, target, sentiment expression, sentiment polarity). \n\nEach tuple should contain:\n\n- Holder: The entity expressing the sentiment, if there is no explicit holder, use \'NULL\' as the holder.\n\n- Target: The entity being evaluated, if there is no explicit target, use \'NULL\' as the target.\n\n- Sentiment Expression: The phrase conveying the sentiment.\n\n- Sentiment Polarity: The polarity of the sentiment, either positive, negative, or neutral.\n\nFollow these rules:\n\n1. If there is no sentiment expression, return \'NULL\' for all fields.\n\n2. Please return python list only, without any other comments or texts.""",\n\n            \'absa/dsunis\':"""Please perform the Structured Sentiment Analysis task. Given a sentence, extract all opinion tuples in the format (holder, target, sentiment expression, sentiment polarity). \n\nEach tuple should contain:\n\n- Holder: The entity expressing the sentiment, if there is no explicit holder, use \'NULL\' as the holder.\n\n- Target: The entity being evaluated, if there is no explicit target, use \'NULL\' as the target.\n\n- Sentiment Expression: The phrase conveying the sentiment.\n\n- Sentiment Polarity: The polarity of the sentiment, either positive, negative, or neutral.\n\nFollow these rules:\n\n1. If there is no sentiment expression, return \'NULL\' for all fields.\n\n2. Please return python list only, without any other comments or texts.""",\n            \n            \'absa/coqe_camera\':"""Please perform the Comparative Opinion Quintuple Extraction task. Given a sentence, extract all opinion tuples in the format (subject, object, comparative aspect, comparative opinion, comparative preference).\n\nEach tuple should contain:\n\n- subject: The first entity being compared.\n\n- object: The second entity being compared.\n\n- comparative aspect: The feature or attribute being compared.\n\n- comparative opinion: The opinion expression indicating the comparative preference between two entities.\n\n- comparative preference: One of the four possible comparative relationships:\n\n    Worse: The subject is worse than the object.\n\n    Equal: The subject is equal to the object.\n\n    Better: The subject is better than the object.\n\n    Different: The subject is different from the object.\n\nIf\xa0any component is missing, use NULL as its placeholder. Please extract all opinion tuples in the format (subject, object, comparative aspect, comparative opinion, comparative preference). Please return python list only, without any other comments or texts.""",\n    "absa/atsa_rest16":"Please perform Aspect Term Sentiment Analysis task. Given the sentence, extract all (aspect term, sentiment polarity) pairs.",\n    "absa/acsa_rest16":"Please perform aspect-level sentiment analysis task. Given the sentence, tag all (aspect category, sentiment) pairs. Aspect category should be selected from [\'ambience general\', \'drinks prices\', \'drinks quality\', \'drinks style_options\', \'food prices\', \'food quality\', \'food style_options\', \'location general\', \'restaurant general\', \'restaurant miscellaneous\', \'restaurant prices\', \'service general\'], and sentiment should be selected from [\'negative\', \'neutral\', \'positive\']. If there are no target-sentiment pairs, return an empty list. Otherwise return a python list of tuples containing two strings in double quotes. Please return python list only, without any other comments or texts.",\n        }\n        self.construct_sample()\n\n    def __len__(self):\n        return len(self.dataset)\n\n\n    def add_prompt(self, dataset_name, data):\n\n        prompt = self.prompts[dataset_name]\n        prompts = [prompt]\n        prompts.append(\'\')\n\n\n        sentence = data[\'sentence\']\n        demos = random.sample(self.demo_pool, k=self.k_shot)\n\n        for demo in demos:\n            prompts.append(f"Sentence: {demo[\'sentence\']}\\nLabel: {demo[\'label_seq\']}\\n")\n\n        \n        prompts.append(f"Sentence: {sentence}\\nLabel:")\n        data[\'prompts\'] = \'\\n\'.join(prompts)\n        data[\'dataset\'] = dataset_name\n\n\n    def make_label_seq(self, dataset_name, data):\n\n        def make_atsa_rest16_seq(example):\n            if \'label_seq\' in example:\n                return example[\'label_seq\']\n\n            atsa_seq = []\n            for aspect in example[\'aspects\']:\n                polarity = aspect[\'polarity\']\n                aspect_term = aspect[\'target\']\n\n                if aspect_term in (\'NULL\', None):\n                    continue\n\n                assert aspect_term in example[\'sentence\']\n\n                loc = (ifnt0(aspect[\'from\']), ifnt0(aspect[\'to\']))\n\n                atsa_seq.append((aspect_term, polarity, loc))\n\n            atsa_seq = list(set(atsa_seq))\n            atsa_seq = sorted(atsa_seq, key=lambda it: it[-1])\n            atsa_seq = [(aspect_term, polarity) for aspect_term, polarity, _ in atsa_seq]\n            example[\'label_seq\'] = atsa_seq\n\n\n        def make_acsa_rest16_seq(example):\n            if \'label_seq\' in example:\n                return example[\'label_seq\']\n\n\n            acsa_seq = []\n            for aspect in example[\'aspects\']:\n                polarity = aspect[\'polarity\']\n                category = aspect[\'category\'].lower().replace(\'#\', \' \')\n\n                category_polarity = (category, polarity)\n                if category_polarity not in acsa_seq:\n                    acsa_seq.append(category_polarity)\n\n            example[\'label_seq\'] = acsa_seq\n\n\n        def make_asqp_seq(example):\n            if \'label_seq\' in example:\n                print(1)\n                return example[\'label_seq\']\n\n\n            asqp_seq = []\n            for quard in example[\'quards\']:\n\n                aspect = quard[\'aspect\']\n                category = quard[\'category\']\n                sentiment = quard[\'sentiment\']\n                opinion = quard[\'opinion\']\n\n\n\n                asqp_quard = (category, aspect, opinion, sentiment)\n                if asqp_quard not in asqp_seq:\n                    asqp_seq.append(asqp_quard)\n\n            example[\'label_seq\'] = asqp_seq\n\n\n\n        def make_ssa_seq(example):\n            if \'label_seq\' in example:\n                return example[\'label_seq\']\n\n            ssa_seq = []\n            for opinion in example[\'opinions\']:\n\n                if len(opinion[\'Source\'][0]) == 0:\n                    holder = \'NULL\'\n                else:\n                    holder = \' AND \'.join(opinion[\'Source\'][0])\n\n                if len(opinion[\'Target\'][0]) == 0:\n                    target = \'NULL\'\n                else:\n                    target = \' AND \'.join(opinion[\'Target\'][0])\n\n                expressions = opinion[\'Polar_expression\'][0]\n                expressions = \' AND \'.join(expressions)\n\n                sentiment = opinion[\'Polarity\'].lower()\n\n                sst_quard = (holder, target, expressions, sentiment)\n                if sst_quard not in ssa_seq:\n                    ssa_seq.append(sst_quard)\n\n            if example[\'opinions\'] == []:\n                ssa_seq.append((\'NULL\', \'NULL\', \'NULL\', \'NULL\'))\n\n            example[\'label_seq\'] = ssa_seq\n\n        def make_multifaced_seq(example):\n            if \'label_seq\' in example:\n                return example[\'label_seq\']\n            example[\'label_seq\'] = example[\'label\'].lower()\n\n        def make_coqe_seq(example):\n            if \'label_seq\' in example:\n                return example[\'label_seq\']\n\n\n            coqe_seq = []\n            for quintuple in example[\'quintuple\']:\n\n                subject = quintuple[\'subject\'] if quintuple[\'subject\'] != \'\' else \'NULL\'\n                object = quintuple[\'object\'] if quintuple[\'object\'] != \'\' else \'NULL\'\n                aspect = quintuple[\'comparative aspect\'] if quintuple[\'comparative aspect\'] != \'\' else \'NULL\'\n                opinion = quintuple[\'comparative opinion\'] if quintuple[\'comparative opinion\'] != \'\' else \'NULL\'\n                preference = quintuple[\'comparative preference\'] if quintuple[\'comparative preference\'] != \'\' else \'NULL\'\n\n\n                coqe_quard = (subject, object, aspect, opinion, preference)\n                if coqe_quard not in coqe_seq:\n                    coqe_seq.append(coqe_quard)\n\n            example[\'label_seq\'] = coqe_seq\n\n\n\n\n        if dataset_name == \'absa/asqp_rest16\' or dataset_name == \'absa/asqp_laptop\':\n            make_asqp_seq(data)\n        elif dataset_name == \'absa/dsunis\' or dataset_name == \'absa/opener\':\n            make_ssa_seq(data)\n        elif dataset_name == \'absa/coqe_camera\':\n            make_coqe_seq(data)\n        elif dataset_name == \'absa/atsa_rest16\':\n            make_atsa_rest16_seq(data)\n        elif dataset_name == \'absa/acsa_rest16\':\n            make_acsa_rest16_seq(data)\n        else:\n            make_multifaced_seq(data)\n\n\n    def construct_sample(self):\n\n\n        print(self.dataset_name)\n        for item in self.demo_pool:\n            self.make_label_seq(self.dataset_name, item)\n\n\n        for dataset_name, dataset in self.dataset_dict.items():\n            for data in dataset:\n                self.add_prompt(dataset_name, data)\n                self.make_label_seq(dataset_name, data)\n\n\n        self.dataset = []\n        for dataset in self.dataset_dict.values():\n            self.dataset.extend(dataset)\n        self.dataset = self.dataset\n\n\n\n    def __getitem__(self, idx):\n        item = self.dataset[idx]\n        prompt = item[\'prompts\']\n        label_seq = item[\'label_seq\']\n\n        return item\n\n\n\ndef cal_metric(results, f1, seed, subname, model_name_or_path, dataset, output_dir, k_shot, metadata=None):\n    model_name = model_name_or_path.split(\'/\')[-1]\n    print(model_name)\n    current_time = time.strftime(\'%Y_%m_%d_%H_%M_%S\')\n    file_name = f\'{subname}_{model_name}_{dataset}_{seed}_numOfshot{k_shot}_{current_time}\'\n    output_path = os.path.join(output_dir, file_name + \'.json\')\n    print(output_path)\n\n\n\n\n    save_result(output_dir, file_name, f1, 0, metadata)\n    save_json(results, output_path)\n\n\nclass DataCollator:\n    def __init__(self, tokenizer, max_seq_length, mode):\n        self.tokenizer = tokenizer\n        self.max_seq_length = max_seq_length\n        self.mode = mode\n\n    def tok(self, text, max_seq_length):\n        kwargs = {\n            \'text\': text,\n            \'return_tensors\': \'pt\',\n        }\n\n        if max_seq_length in (-1, \'longest\'):\n            kwargs[\'padding\'] = True\n\n        else:\n            kwargs[\'max_length\'] = max_seq_length\n            kwargs[\'padding\'] = True\n            kwargs[\'truncation\'] = True\n\n        batch_encodings = self.tokenizer(**kwargs)\n        return batch_encodings\n\n    def __call__(self, examples):\n\n        texts = [example[\'prompts\'] for example in examples]\n        batch_encodings = self.tok(texts, self.max_seq_length)\n\n        input_ids = batch_encodings[\'input_ids\']\n        attention_mask = batch_encodings[\'attention_mask\']\n\n        return {\n            \'input_ids\': input_ids,\n            \'attention_mask\': attention_mask,\n            \'examples\': examples\n        }\n\ndef get_dataloader(test_examples, mode, batch_size, shuffle, tokenizer):\n    dataloader = DataLoader(\n        dataset=test_examples,\n        batch_size=batch_size,\n        shuffle=shuffle,\n        pin_memory=True,\n        num_workers=0,\n        collate_fn=DataCollator(\n            tokenizer=tokenizer,\n            max_seq_length=max_seq_length,\n            mode=mode,\n        )\n    )\n\n    print(\'dataloader-\' + mode, len(dataloader))\n    return dataloader\n\n\ndef test_dataloader(test_examples, eval_batch_size,tokenizer):\n    return get_dataloader(test_examples, "test", eval_batch_size, shuffle=False, tokenizer=tokenizer)\n\n\n# 从test.json 和 train.json 中加载数据\ndef load_dataset(data_path, demo_split=\'train\'):\n    test_file_name = os.path.join(data_path, \'test.json\')\n    test_examples = load_json(test_file_name)\n\n    demo_pool_file_name = os.path.join(data_path, demo_split + \'.json\')\n    demo_examples = load_json(demo_pool_file_name)\n\n    raw_datasets = {\n        \'test\': test_examples,\n        \'demo\': demo_examples\n    }\n\n    print(\'-----------data statistic-------------\')\n    for mode in (\'test\',\'demo\'):\n        num_samples = len(raw_datasets[mode])\n        print(f\'{mode.upper():<5} | Samples: {num_samples:<5}\')\n    print(\'--------------------------------------\')\n\n\n\n    return raw_datasets\n\n\n\nimport sys\nimport argparse\nimport datetime\nif __name__ == \'__main__\':\n    print(\'hello\')\n    parser = argparse.ArgumentParser()\n\n    parser.add_argument(\'--dataset\', required=True, )\n    parser.add_argument(\'--subname\', required=True, )\n    parser.add_argument(\'--seed\', required=True, type=int)\n    parser.add_argument(\'--model_name_or_path\', default=None )\n    parser.add_argument(\'--model_version\', default=None)\n    parser.add_argument(\'--max_seq_length\', type=int, default=2048)\n    parser.add_argument(\'--data_prop\', type=float, default=1)\n    parser.add_argument(\'--eval_batch_size\', type=int, default=1)\n    parser.add_argument(\'--data_dir\', default=os.path.join(os.path.dirname(__file__), \'datasets\'))\n    parser.add_argument(\'--output_dir\', default=\'./output\')\n    parser.add_argument(\'--k_shot\', type=int, default=4)\n\n    add_runtime_arguments(parser)\n    args = parser.parse_args()\n    prepare_runtime(args)\n\n    dataset = args.dataset\n    subname = args.subname\n    seed = args.seed\n    model_version = args.model_version\n    model_name_or_path = args.model_name_or_path\n    max_seq_length = args.max_seq_length\n    data_prop = args.data_prop\n    eval_batch_size = args.eval_batch_size\n    data_dir = args.data_dir\n    output_dir = args.output_dir\n    k_shot = args.k_shot\n\n    data_path = os.path.join(data_dir, dataset)\n    torch.manual_seed(seed)\n    random.seed(seed)\n\n    torch.cuda.manual_seed(seed)\n\n\n    model_name_or_path = args.model_name_or_path\n    print(\'model name: \', model_name_or_path)\n    print(\'-\'*20+\'Experiment Setup\'+\'-\'*20)\n    print(\'data_path: \', data_path)\n    print(f\' subname: {subname}\\n dataset: {dataset}\\n seed:{seed}\\n model_name_or_path:{model_name_or_path}\\n data_path:{data_path}\\n output_dir:{output_dir}\\n eval_batch_size:{eval_batch_size}\\n data_prop:{data_prop}\\n max_seq_length:{max_seq_length}\\n numbers_of_shot:{k_shot}\')\n    print(\'-\' * 50)\n\n\n    raw_datasets = select_examples(load_dataset(data_path, args.demo_split), args)\n\n\n\n    test_examples = FineTuneDataset(dataset, raw_datasets[\'test\'], raw_datasets[\'demo\'], k_shot, model_name_or_path, \'test\', max_seq_length)\n    \n\n    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, padding_side=\'left\')\n    tokenizer.pad_token = tokenizer.eos_token\n\n\n    \n    model = load_model(args)\n    configure_tokenizer(tokenizer, model, args, args.max_new_tokens)\n    max_seq_length = args.max_seq_length\n    truncated = limit_prompts(test_examples, tokenizer, max_seq_length)\n    metadata = run_metadata(args, raw_datasets, truncated)\n    cache = PredictionCache(args.resume_file, metadata)\n    pending = [example for example in test_examples if cache.get(example) is None]\n    dataloader = test_dataloader(pending, eval_batch_size, tokenizer)\n\n\n    print(model_name_or_path)\n    model.eval()\n\n    with torch.inference_mode():\n        for batch in tqdm(dataloader):\n            input_ids = batch[\'input_ids\'].to(model.device)\n            generated_ids = model.generate(\n                input_ids,\n                attention_mask=batch[\'attention_mask\'].to(model.device),\n                max_new_tokens=args.max_new_tokens,\n                num_beams=1, num_return_sequences=1, do_sample=False,\n                pad_token_id=tokenizer.pad_token_id,\n            )\n            # Decode only new tokens, independent of prompt truncation and shot count.\n            generateds = tokenizer.batch_decode(\n                generated_ids[:, input_ids.shape[1]:], skip_special_tokens=True)\n            for example, prediction in zip(batch[\'examples\'], generateds):\n                cache.put(example, prediction.strip())\n            cache.flush()\n            del generated_ids, input_ids\n\n    outputs = [(example, cache.get(example)) for example in test_examples]\n\n    current_results = []\n\n    if  dataset == \'absa/asqp_rest16\' or dataset == \'absa/asqp_laptop\': \n        current_results.append(Result_asqp.parse_from(outputs))\n    elif dataset == \'absa/opener\' or dataset == \'absa/dsunis\':\n        current_results.append(Result_ssa.parse_from(outputs))\n    elif dataset == \'absa/coqe_camera\':\n        current_results.append(Result_coqe.parse_from(outputs))\n    elif dataset == \'absa/atsa_rest16\':\n        current_results.append(Result_atsa.parse_from(outputs))\n    elif dataset == \'absa/acsa_rest16\':\n        current_results.append(Result_acsa.parse_from(outputs))\n    else:\n        raise ValueError(f\'No dataset named {dataset}\')\n\n\n    for result in current_results:\n        result.cal_metric()\n\n\n\n\n    current_results[0].report()\n    f1_scores = current_results[0].monitor\n    print(f1_scores)\n    \n    \n    \n    cal_metric(outputs,f1_scores, seed, subname, model_name_or_path, dataset, output_dir, k_shot, metadata)\n', 'notebook_runtime.py': '"""Shared, memory-conscious evaluation support for hosted notebooks."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport warnings\n\n\ndef add_runtime_arguments(parser):\n    parser.add_argument(\'--dtype\', choices=[\'auto\', \'float16\', \'bfloat16\', \'float32\'], default=\'auto\')\n    parser.add_argument(\'--load_in_4bit\', action=\'store_true\')\n    parser.add_argument(\'--max_samples\', type=int, default=0, help=\'0: all; otherwise first N test examples\')\n    parser.add_argument(\'--demo_split\', choices=[\'train\', \'val\'], default=\'train\')\n    parser.add_argument(\'--max_new_tokens\', type=int, default=150)\n    parser.add_argument(\'--resume_file\', help=\'Atomic prediction checkpoint; use a persistent output path\')\n    parser.add_argument(\'--adapter\', default=None, help=\'Path to PEFT adapter checkpoint directory\')\n\n\ndef prepare_runtime(args):\n    import torch\n    if args.eval_batch_size < 1 or args.k_shot < 0 or args.max_samples < 0:\n        raise ValueError(\'batch_size must be positive; k_shot and max_samples must be nonnegative\')\n    if args.max_seq_length != -1 and args.max_seq_length < 1:\n        raise ValueError(\'max_seq_length must be -1 or a positive token count\')\n    if not 0 < args.data_prop <= 1 or args.max_new_tokens < 1:\n        raise ValueError(\'data_prop must be in (0, 1]; max_new_tokens must be positive\')\n    if args.model_name_or_path:\n        model_path = args.model_name_or_path\n    else:\n        json_file = Path(__file__).with_name(\'model_name_kaggle.json\')\n        if not json_file.exists():\n            json_file = Path(__file__).with_name(\'model_name.json\')\n        mapping = json.loads(json_file.read_text())\n        model_path = mapping.get(args.model_version, args.model_version)\n        if not model_path:\n            raise ValueError(\'Provide --model_name_or_path or --model_version\')\n        if model_path.startswith(\'.\'):\n            model_path = str(Path(__file__).parent / model_path)\n    args.model_name_or_path = model_path\n    if args.load_in_4bit and not torch.cuda.is_available():\n        raise ValueError(\'4-bit evaluation requires an available CUDA GPU\')\n    dtype = args.dtype\n    if dtype == \'auto\':\n        dtype = (\'bfloat16\' if torch.cuda.is_bf16_supported() else \'float16\') if torch.cuda.is_available() else \'float32\'\n    if dtype == \'bfloat16\' and (not torch.cuda.is_available() or not torch.cuda.is_bf16_supported()):\n        raise ValueError(\'This GPU does not support BF16; use --dtype auto or float16\')\n    if not torch.cuda.is_available() and dtype == \'float16\':\n        raise ValueError(\'Use float32 for CPU evaluation\')\n    args.resolved_dtype = dtype\n    Path(args.output_dir).mkdir(parents=True, exist_ok=True)\n    print(f\'Runtime: {"CUDA" if torch.cuda.is_available() else "CPU"}, dtype={dtype}, 4bit={args.load_in_4bit}\')\n    return model_path\n\n\ndef _disable_torchao():\n    import sys\n    try:\n        import peft.import_utils\n        peft.import_utils.is_torchao_available = lambda: False\n    except Exception:\n        pass\n    try:\n        import peft.tuners.lora.torchao\n        peft.tuners.lora.torchao.is_torchao_available = lambda: False\n        peft.tuners.lora.torchao.dispatch_torchao = lambda *args, **kwargs: None\n    except Exception:\n        pass\n    for mod in list(sys.modules.values()):\n        if hasattr(mod, \'is_torchao_available\'):\n            try:\n                mod.is_torchao_available = lambda: False\n            except Exception:\n                pass\n\n\ndef load_model(args):\n    import torch\n    from transformers import AutoModelForCausalLM, BitsAndBytesConfig\n    options = dict(torch_dtype=getattr(torch, args.resolved_dtype), low_cpu_mem_usage=True)\n    if torch.cuda.is_available():\n        # One GPU avoids accidental CPU offload or combining incompatible notebook GPUs.\n        options[\'device_map\'] = {\'\': 0}\n    if args.load_in_4bit:\n        options[\'quantization_config\'] = BitsAndBytesConfig(\n            load_in_4bit=True, bnb_4bit_quant_type=\'nf4\',\n            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=getattr(torch, args.resolved_dtype))\n    model = AutoModelForCausalLM.from_pretrained(args.model_name_or_path, **options)\n    args.resolved_model_revision = getattr(model.config, \'_commit_hash\', None)\n    if getattr(args, \'adapter\', None):\n        _disable_torchao()\n        from peft import PeftModel\n        print(f\'Loading PEFT adapter from {args.adapter}\')\n        model = PeftModel.from_pretrained(model, args.adapter)\n    model.eval()\n    return model\n\n\ndef select_examples(raw, args):\n    count = max(1, int(len(raw[\'test\']) * args.data_prop)) if raw[\'test\'] else 0\n    if args.max_samples:\n        count = min(count, args.max_samples)\n    raw[\'test\'] = raw[\'test\'][:count]\n    if not raw[\'test\']:\n        raise ValueError(\'Test split is empty\')\n    if args.k_shot > len(raw[\'demo\']):\n        raise ValueError(f\'k_shot={args.k_shot} exceeds demo pool ({len(raw["demo"])})\')\n    return raw\n\n\ndef configure_tokenizer(tokenizer, model, args, output_tokens):\n    tokenizer.padding_side = \'left\'\n    tokenizer.truncation_side = \'left\'  # Keep the final question and Label: suffix.\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    if tokenizer.pad_token_id is None:\n        raise ValueError(\'Tokenizer needs a pad or EOS token\')\n    limits = [getattr(tokenizer, \'model_max_length\', 10**9),\n              getattr(model.config, \'max_position_embeddings\', 10**9)]\n    context = min(int(v) for v in limits if v is not None)\n    available = context - output_tokens\n    if available < 1:\n        raise ValueError(\'Generation budget exceeds the model context window\')\n    if args.max_seq_length == -1:\n        args.max_seq_length = available\n    else:\n        args.max_seq_length = min(args.max_seq_length, available)\n\n\ndef limit_prompts(examples, tokenizer, max_length):\n    """Truncate demonstrations first; reject loss of the actual test sentence."""\n    truncated = 0\n    for example in examples:\n        text = example[\'prompts\']\n        # The final prompt suffix contains the complete test input.\n        suffix = f"Sentence: {example[\'sentence\']}\\nLabel:"\n        if len(tokenizer(suffix, add_special_tokens=False)[\'input_ids\']) + tokenizer.num_special_tokens_to_add() > max_length:\n            raise ValueError(\'Test sentence exceeds max_seq_length. Increase the limit; refusing to silently drop test text.\')\n        ids = tokenizer(text, add_special_tokens=True)[\'input_ids\']\n        if len(ids) > max_length:\n            truncated += 1\n    if truncated:\n        warnings.warn(f\'{truncated}/{len(examples)} prompts exceed {max_length} tokens. Left truncation removes instructions/demos; results differ from untruncated evaluation.\')\n    return truncated\n\n\ndef run_metadata(args, raw, truncated):\n    from importlib.metadata import version\n    settings = {k: getattr(args, k) for k in (\n        \'model_name_or_path\', \'k_shot\', \'max_seq_length\', \'max_samples\', \'data_prop\',\n        \'demo_split\', \'resolved_dtype\', \'load_in_4bit\', \'max_new_tokens\', \'eval_batch_size\')}\n    fingerprint = hashlib.sha256(json.dumps(raw, sort_keys=True).encode()).hexdigest()\n    source_files = [Path(__file__), Path(__file__).with_name(\'fsa.py\'),\n                    Path(__file__).with_name(\'icl_multifaced.py\'), Path(__file__).with_name(\'utils.py\')]\n    source_files += sorted(Path(__file__).parent.glob(\'parse_utils/*.py\'))\n    source_hash = hashlib.sha256(b\'\'.join(p.read_bytes() for p in source_files)).hexdigest()\n    return dict(experiment=args.subname, dataset=args.dataset, seed=args.seed,\n                settings=settings, data_sha256=fingerprint, num_samples=len(raw[\'test\']),\n                source_sha256=source_hash, model_revision=args.resolved_model_revision,\n                truncated_prompts=truncated,\n                versions={p: version(p) for p in (\'torch\', \'transformers\', \'accelerate\', \'numpy\', \'scikit-learn\')})\n\n\nclass PredictionCache:\n    """Atomic batch checkpoints; validate data/settings before reusing predictions."""\n    def __init__(self, path, metadata):\n        self.path = Path(path) if path else None\n        self.metadata = metadata\n        self.entries = {}\n        if self.path and self.path.exists():\n            saved = json.loads(self.path.read_text(encoding=\'utf-8\'))\n            if saved[\'metadata\'] != metadata:\n                raise ValueError(\'Checkpoint settings/data/environment differ. Choose a new resume_file or restore the original configuration.\')\n            self.entries = saved[\'entries\']\n            print(f\'Resuming {len(self.entries)} cached predictions\')\n\n    @staticmethod\n    def key(example):\n        return hashlib.sha256(example[\'prompts\'].encode()).hexdigest()\n\n    def get(self, example):\n        return self.entries.get(self.key(example))\n\n    def put(self, example, value):\n        self.entries[self.key(example)] = value\n\n    def flush(self):\n        if self.path:\n            self.path.parent.mkdir(parents=True, exist_ok=True)\n            temp = self.path.with_suffix(self.path.suffix + \'.tmp\')\n            temp.write_text(json.dumps({\'metadata\': self.metadata, \'entries\': self.entries}), encoding=\'utf-8\')\n            os.replace(temp, self.path)\n\n\ndef save_result_kaggle(output_dir, exper_name, f1, acc, metadata=None):\n    result = (exper_name, {\'f1\': round(f1, 4), \'acc\': round(acc, 4)})\n    if metadata is not None:\n        result[1][\'metadata\'] = metadata\n    os.makedirs(output_dir, exist_ok=True)\n    with open(os.path.join(output_dir, \'result.txt\'), \'a\', encoding=\'utf-8-sig\') as f:\n        f.write(json.dumps(result) + \'\\n\')\n', 'model_name_kaggle.json': '{\n  "qwen2.5-0.5b": "Qwen/Qwen2.5-0.5B-Instruct",\n  "qwen3-0.6b": "Qwen/Qwen3-0.6B",\n  "qwen3.5-0.8b": "Qwen/Qwen3.5-0.8B",\n  "qwen2.5-1.5b": "Qwen/Qwen2.5-1.5B-Instruct",\n  "qwen3-1.7b": "Qwen/Qwen3-1.7B",\n  "smollm2-360m": "HuggingFaceTB/SmolLM2-360M-Instruct",\n  "smollm2-1.7b": "HuggingFaceTB/SmolLM2-1.7B-Instruct",\n  "deepseek-r1-1.5b": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",\n  "Qwen2.5-1.5B-sentiment-distillation-v1": "./model_weights/Qwen2.5-1.5B-sentiment-distillation-v1",\n  "llama-3-1B-sentiment-distillation-v1": "./model_weights/llama-3-1B-sentiment-distillation-v1",\n  "llama-3-3B-sentiment-distillation-v1": "./model_weights/llama-3-3B-sentiment-distillation-v1"\n}\n'}
for relative, source in BUNDLED.items():
    target = BASE / 'source' / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
SCRIPT = BASE / 'source/training/train_distill.py'
REQUIREMENTS = BASE / 'source/requirements-train-kaggle.txt'

def run(arguments, log=None):
    command = [sys.executable, str(SCRIPT), *map(str, arguments)]
    if log is None:
        subprocess.run(command, check=True)
        return
    # Stream output and persist it; failed child raises instead of continuing to next mode.
    with Path(log).open('a', encoding='utf-8') as stream:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                stream.write(line)
                stream.flush()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)


## Dependency trên Kaggle

Cell cài thư viện train vào runtime hiện tại; không nâng/hạ PyTorch/CUDA.
Pip dry-run dừng nếu kế hoạch thay `torch`, `torchvision`, `torchaudio`, `triton`, hoặc `nvidia-*`.
Mọi code model chạy ở subprocess mới, tránh module cache của notebook sau pip install.
Không cần Hugging Face token cho Qwen public; nếu bị rate-limit, dùng secret `HF_TOKEN`.


In [ ]:
if INSTALL_DEPENDENCIES:
    # Gỡ bỏ torchao lỗi thời của Kaggle để tránh xung đột với peft
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
    extra = ['bitsandbytes==0.46.1'] if FOUR_BIT else []
    report = WORK / 'pip-plan.json'
    base = [sys.executable, '-m', 'pip', 'install', '--only-binary=:all:', '-r', str(REQUIREMENTS), *extra]
    subprocess.run([*base, '--dry-run', '--report', str(report)], check=True)
    plan = json.loads(report.read_text())
    names = [p['metadata']['name'].lower().replace('_', '-') for p in plan.get('install', [])]
    forbidden = [n for n in names if n in {'torch', 'torchvision', 'torchaudio', 'triton'} or n.startswith('nvidia-')]
    if forbidden:
        raise RuntimeError(f'Pip would replace CUDA stack: {forbidden}; use a compatible platform runtime.')
    subprocess.run(base, check=True)
else:
    print('Using platform packages; the next runtime cell checks imports.')


In [ ]:
# GPU/dependency validation only; no synthetic model inference.
subprocess.run([sys.executable, '-c', r"""
import json, torch, transformers, peft, accelerate
from importlib.metadata import version
from packaging.version import Version
assert Version(transformers.__version__) >= Version('4.51.0'), 'Qwen3 requires transformers >= 4.51'
assert torch.cuda.is_available(), 'Enable a GPU accelerator'
print(json.dumps({
    'versions': {p: version(p) for p in ('torch', 'transformers', 'peft', 'accelerate', 'huggingface-hub')},
    'gpus': [{'name': torch.cuda.get_device_name(i),
              'vram_gib': torch.cuda.get_device_properties(i).total_memory / 2**30}
             for i in range(torch.cuda.device_count())],
    'bf16': torch.cuda.is_bf16_supported(),
}, indent=2))
"""], check=True)
print('Pilot dùng GPU 0; không tự gộp VRAM của hai T4.')


## Tải đúng file và audit

Dùng `hf_hub_download` cho từng file, tránh lỗi schema khi loader trộn ba teacher.
Revision được khóa; file attach thủ công vẫn được kiểm SHA-256 ở prepare nhưng cần tự xác minh provenance.
Audit lấy mẫu reservoir trải khắp file; không chỉ xem những dòng đầu thường cùng task.


In [ ]:
from huggingface_hub import HfApi, hf_hub_download
if LOCAL_TEACHER_FILE:
    CORPUS = Path(LOCAL_TEACHER_FILE)
    CORPUS_REVISION = 'local-unverified-provenance:' + DATASET_REVISION
else:
    CORPUS_REVISION = HfApi().dataset_info(DATASET, revision=DATASET_REVISION).sha
    CORPUS = Path(hf_hub_download(DATASET, TEACHER_FILE, repo_type='dataset', revision=CORPUS_REVISION))
run(['audit', '--file', CORPUS, '--group_field', 'selected_cluster_id', '--examples', 40,
     '--output', WORK / 'audit.json'], log=WORK / 'audit.log')


## Xác nhận field mapping sau khi đọc audit

Đọc tối thiểu 30–50 mẫu: user text đã nằm trong `instruction` chưa? `input` rỗng có nghĩa gì?
`PROMPT_FIELDS` là thứ tự nối field bằng hai newline; không được nhân đôi user text hoặc demonstration.
Ví dụ **chỉ khi đã xác minh**: `['instruction', 'text']` nếu instruction không chứa câu hỏi cuối;
hoặc `['instruction', 'input']` nếu input là phần cần nối; hoặc `['instruction']` nếu đã chứa đầy đủ.
Giữ nguyên field gốc và đáp án. Không parse/viết lại response teacher.

`SOURCE_FIELD` phải là văn bản nguồn dùng phát hiện cùng input qua nhiều task; không dùng task_name/ID.
`GROUP_FIELD` cần đủ mẫu/nhóm; `cluster_id` có thể rỗng, kiểm tra `selected_cluster_id` trong audit.
Đặt `MAPPING_CONFIRMED=True` sau khi kiểm tra. Đây là bước chặn train sai schema, không tự suy đoán.


In [ ]:
PROMPT_FIELDS = ['instruction', 'text']
RESPONSE_FIELD = 'Qwen/Qwen3-32B_output'
GROUP_FIELD = 'selected_cluster_id'
SOURCE_FIELD = 'text'
MAPPING_CONFIRMED = True

if not MAPPING_CONFIRMED or not PROMPT_FIELDS:
    raise ValueError('Read audit.json, set PROMPT_FIELDS and confirm mapping before prepare.')
PREPARED = WORK / 'prepared'
prepare_args = ['prepare', '--file', CORPUS, '--output', PREPARED, '--model', MODEL,
                '--model_revision', MODEL_REVISION, '--corpus_revision', CORPUS_REVISION,
                '--prompt_fields', ','.join(PROMPT_FIELDS), '--response_field', RESPONSE_FIELD,
                '--group_field', GROUP_FIELD, '--source_field', SOURCE_FIELD, '--mapping_confirmed',
                '--max_length', MAX_LENGTH, '--max_train', MAX_TRAIN, '--seed', SEED,
                '--max_groups', 16, '--probe_per_group', 4, '--holdout_per_group', 4]
request = dict(model=MODEL, model_revision=MODEL_REVISION, corpus_revision=CORPUS_REVISION,
               file=str(CORPUS), prompt_fields=PROMPT_FIELDS, response_field=RESPONSE_FIELD,
               group_field=GROUP_FIELD, source_field=SOURCE_FIELD, max_length=MAX_LENGTH,
               max_train=MAX_TRAIN, seed=SEED)
request_file = WORK / 'prepare-request.json'
if (PREPARED / 'manifest.json').exists():
    if not request_file.exists() or json.loads(request_file.read_text()) != request:
        raise ValueError('Prepare settings differ. Change RUN_TAG instead of reusing old splits.')
    print('Reusing prepared corpus; train checks split hashes.')
else:
    run(prepare_args, log=WORK / 'prepare.log')
    request_file.write_text(json.dumps(request, indent=2))
manifest = json.loads((PREPARED / 'manifest.json').read_text())
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print((PREPARED / 'preview.json').read_text())


## Train baseline và đề xuất

Uniform và progress dùng cùng samples, shuffle, seed, LoRA, steps, response-only mean-per-example loss.
Mọi nhánh đều đo probe/holdout cùng lịch để kiểm soát chi phí đo. Progress cập nhật trọng số bằng EMA
của tiến bộ tương đối trên probe, hỗn hợp với trọng số đều; giới hạn 0,5–2 và chuẩn hóa theo train counts.
Probe được dùng điều phối; holdout tách input độc lập và chỉ ghi log, không cập nhật controller.
Không chọn cấu hình bằng SentiBench-test. Nhóm không có probe vẫn giữ trọng số nền; xem coverage.

Checkpoint gồm adapter, optimizer, scheduler, scaler, RNG, data cursor và controller.
Để resume: restore toàn thư mục WORK vào cùng đường dẫn, giữ source/config/packages và điền RESUME.
Không chỉ restore adapter. Hai nhánh mặc định 200 bước vẫn chỉ là pilot, chưa đi qua toàn train pool.


In [ ]:
for mode in MODES:
    output = WORK / mode
    run_request = dict(model=MODEL, prepared=str(PREPARED), mode=mode, steps=STEPS,
                       batch_size=1, grad_accum=GRAD_ACCUM, learning_rate=LEARNING_RATE,
                       rank=16, seed=SEED, probe_every=PROBE_EVERY,
                       warmup_steps=PROBE_EVERY, save_every=SAVE_EVERY, four_bit=FOUR_BIT)
    request_path = WORK / f'{mode}-request.json'
    if request_path.exists() and json.loads(request_path.read_text()) != run_request:
        raise ValueError('Train settings changed. Choose a new RUN_TAG; do not silently skip an old run.')
    request_path.write_text(json.dumps(run_request, indent=2))
    if (output / 'summary.json').exists():
        print('Completed already:', mode, '— choose a new RUN_TAG to rerun with changed settings.')
        continue
    arguments = ['train', '--prepared', PREPARED, '--output', output, '--mode', mode,
                 '--steps', STEPS, '--batch_size', 1, '--grad_accum', GRAD_ACCUM,
                 '--learning_rate', LEARNING_RATE, '--rank', 16, '--seed', SEED,
                 '--probe_every', PROBE_EVERY, '--warmup_steps', PROBE_EVERY,
                 '--save_every', SAVE_EVERY]
    if FOUR_BIT:
        arguments.append('--four_bit')
    if RESUME.get(mode):
        arguments += ['--resume', RESUME[mode]]
    run(arguments, log=WORK / f'{mode}.log')


## Số liệu pilot

CE là mức khớp đáp án teacher, **không phải Accuracy/Macro-F1/SentiBench score**.
Thời gian gồm train, probe, holdout và các checkpoint trước đó; VRAM ghi peak tensor allocated,
không thay cho tổng VRAM hiển thị bởi nvidia-smi. So token và optimizer_updates giữa các nhánh;
FP16 có thể bỏ optimizer update khi overflow — nếu khác nhiều cần chỉnh LR/precision và chạy lại.
Để tuyên bố cải thiện chất lượng, phải đánh giá checkpoint trên benchmark với evaluator đã kiểm chứng.


In [ ]:
rows = []
for mode in MODES:
    path = WORK / mode / 'summary.json'
    if not path.exists():
        continue
    summary = json.loads(path.read_text())
    state = summary['state']
    events = [json.loads(s) for s in (WORK / mode / 'metrics.jsonl').read_text().splitlines() if s.strip()]
    measured = [r for r in events if r.get('holdout_loss')]
    counts = manifest['holdout_groups']
    def holdout_mean(event):
        values = event['holdout_loss']
        return sum(values[g] * counts[g] for g in values) / sum(counts[g] for g in values)
    rows.append({'mode': mode, 'steps': state['step'], 'updates': state['optimizer_updates'],
                 'initial_holdout_ce': holdout_mean(measured[0]),
                 'final_holdout_ce': holdout_mean(measured[-1]),
                 'train_tokens': state['train_tokens'], 'probe_tokens': state['probe_tokens'],
                 'holdout_tokens': state['holdout_tokens'],
                 'minutes': state['elapsed_seconds'] / 60,
                 'peak_allocated_gib': state['peak_vram_bytes'] / 2**30})
print(json.dumps(rows, indent=2))
(WORK / 'comparison.json').write_text(json.dumps(rows, indent=2))


## Export model và lưu artifacts

Adapter không phải model đầy đủ để truyền trực tiếp cho evaluator hiện tại. Nếu cần, đặt EXPORT_MODE
rồi merge trên CPU; cần RAM cho model float32 và dung lượng đĩa. Export từng nhánh khi dùng.
Sau đó dùng đường dẫn `merged-*` làm `--model_name_or_path` trong pipeline đánh giá đã audit.
Evaluator cũ chưa có chat/non-thinking wrapper và chưa xác nhận giao thức v2; không dùng điểm của nó
để kết luận cải thiện theo paper trước khi kiểm chứng. Notebook này chưa tích hợp SentiBench evaluator v2.

Save Version với outputs trên Kaggle. File ZIP nhỏ gồm log/manifest/split IDs/config/adapter, không gồm
base model, raw corpus, tokenized splits hay optimizer. Resume cần giữ cả WORK bằng Kaggle outputs.
Trước khi xóa session, kiểm tra output thực sự được lưu. Không có đảm bảo session đủ cho full run.


In [ ]:
if EXPORT_MODE:
    if EXPORT_MODE not in MODES:
        raise ValueError('EXPORT_MODE must be a trained mode')
    run(['export', '--run', WORK / EXPORT_MODE, '--output', WORK / f'merged-{EXPORT_MODE}'],
        log=WORK / f'export-{EXPORT_MODE}.log')


## Đánh giá So sánh Model gốc (Before) vs Model sau Finetuning (After)

Cell sau tự động chạy đánh giá trên bài kiểm thử chuẩn (mặc định `sc/imdb`, `sc/sst2`, 4-shot)
để đo lường mức độ cải thiện trực tiếp của LoRA adapter so với mô hình gốc trước khi finetune.
Kết quả được xuất ra dạng bảng đối chứng Before vs After và lưu tại `evaluation_comparison.json`.


In [ ]:
# Đánh giá Before vs After
EVAL_DATASETS = 'sc/imdb,sc/sst2'
EVAL_BATCH_SIZE = 4
EVAL_K_SHOT = 4
ADAPTER_PATH = WORK / 'uniform' / 'adapter'

eval_script = BASE / 'source/kaggle_evaluate.py'
if ADAPTER_PATH.exists() and eval_script.exists():
    eval_cmd = [
        sys.executable, str(eval_script),
        '--model', MODEL,
        '--adapter', str(ADAPTER_PATH),
        '--datasets', EVAL_DATASETS,
        '--k_shot', str(EVAL_K_SHOT),
        '--eval_batch_size', str(EVAL_BATCH_SIZE),
        '--output_dir', str(WORK / 'eval_comparison')
    ]
    print("=== Đang chạy đánh giá so sánh Trước vs Sau Finetuning ===")
    subprocess.run(eval_cmd, check=True)
else:
    print(f"Adapter chưa sẵn sàng tại {ADAPTER_PATH}; hãy hoàn tất bước train trước.")


In [ ]:
# Portable run metadata and adapters; full checkpoints stay in WORK for Save Version.
import zipfile
archive = BASE / f'{RUN_TAG}-{MODEL_TAG}-artifacts.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in WORK.rglob('*'):
        if not path.is_file():
            continue
        relative = path.relative_to(WORK)
        if any(p.startswith('checkpoint-') or p.startswith('merged-') for p in relative.parts):
            continue
        if relative.parts[0] == 'prepared' and path.name not in {
            'manifest.json', 'preview.json', 'rejected.json'}:
            continue
        z.write(path, relative)
    for name in ('train', 'probe', 'holdout'):
        ids = []
        with (PREPARED / f'{name}.jsonl').open() as stream:
            for line in stream:
                r = json.loads(line)
                ids.append({k: r[k] for k in ('id', 'source_key', 'group', 'train_group')})
        z.writestr(f'prepared/{name}-ids.json', json.dumps(ids))
    for relative, source in BUNDLED.items():
        z.writestr('source/' + relative, source)
print('Artifacts:', archive)
print('Full resume checkpoints:', WORK)


Nguồn: [corpus v2](https://huggingface.co/datasets/zhang-yice/sentiment-distillation-v2),
[CompEffDist](https://arxiv.org/html/2510.24425v1),
[Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B).
Phương pháp progress-weighting là ứng viên nghiên cứu; chưa chứng nhận novelty hay lợi ích.
